# 1. Simple Block Model

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# --- CONFIGURATION ---
N = 60                   # Small grid for intuition
VOL_FRAC_SN = 0.49       # 49% Volume Fraction (Corrected)
RHO_SN = 18.0            # Tin
RHO_BI = 129.0           # Bismuth
R_INT_VAL = 45.0         # Interface Penalty

def solve_resistor_network_corrected(grid, rho_A, rho_B, r_interface):
    """
    Solves the resistor network with CORRECT INSULATING BOUNDARIES.
    (Previous version had 'leaky' top/bottom walls).
    """
    ny, nx = grid.shape
    V = np.zeros((ny, nx))

    # Linear guess to speed up convergence
    for x in range(nx): V[:, x] = 1.0 - x/(nx-1)

    # Fix Left/Right Voltage
    V[:, 0] = 1.0
    V[:, -1] = 0.0

    rho_map = np.where(grid == 0, rho_A, rho_B)

    # Calculate Conductances (G = 1/R)
    # Horizontal
    R_right = (rho_map[:, :-1] + rho_map[:, 1:]) / 2.0
    mask_diff_right = (grid[:, :-1] != grid[:, 1:])
    R_right[mask_diff_right] += r_interface
    G_right = 1.0 / R_right

    # Vertical
    R_down = (rho_map[:-1, :] + rho_map[1:, :]) / 2.0
    mask_diff_down = (grid[:-1, :] != grid[1:, :])
    R_down[mask_diff_down] += r_interface
    G_down = 1.0 / R_down

    print("  > Solving...", end="")
    # Iterative Solver with Boundary Handling
    for i in range(4000):
        if i % 1000 == 0: print(f".", end="")

        # 1. Update Inner Pixels
        # Sum of (V_neighbor * G_neighbor)
        term_right = V[1:-1, 2:] * G_right[1:-1, 1:]
        term_left  = V[1:-1, :-2] * G_right[1:-1, :-1]
        term_down  = V[2:, 1:-1] * G_down[1:, 1:-1]
        term_up    = V[:-2, 1:-1] * G_down[:-1, 1:-1]

        sum_G = (G_right[1:-1, 1:] + G_right[1:-1, :-1] +
                 G_down[1:, 1:-1] + G_down[:-1, 1:-1])

        V[1:-1, 1:-1] = (term_right + term_left + term_down + term_up) / sum_G

        # 2. Update Top Row (Insulating Top: No Up Neighbor)
        term_r_0 = V[0, 2:] * G_right[0, 1:]
        term_l_0 = V[0, :-2] * G_right[0, :-1]
        term_d_0 = V[1, 1:-1] * G_down[0, 1:-1]
        sum_G_0  = G_right[0, 1:] + G_right[0, :-1] + G_down[0, 1:-1]
        V[0, 1:-1] = (term_r_0 + term_l_0 + term_d_0) / sum_G_0

        # 3. Update Bottom Row (Insulating Bottom: No Down Neighbor)
        term_r_last = V[-1, 2:] * G_right[-1, 1:]
        term_l_last = V[-1, :-2] * G_right[-1, :-1]
        term_u_last = V[-2, 1:-1] * G_down[-1, 1:-1]
        sum_G_last  = G_right[-1, 1:] + G_right[-1, :-1] + G_down[-1, 1:-1]
        V[-1, 1:-1] = (term_r_last + term_l_last + term_u_last) / sum_G_last

    print(" Done!")

    # Calculate Current exiting the right side
    V_diff = V[:, -2] - V[:, -1]
    R_link = R_right[:, -1]
    currents = V_diff / R_link
    total_current = np.sum(currents)

    if total_current == 0: return float('inf'), V
    rho_eff = (1.0 / total_current) # For square grid, rho = R
    return rho_eff, V

# --- MAIN RUN ---
print("Running Corrected Intuition Models...")

# 1. Parallel (Ideal)
grid_par = np.ones((N, N))
num_sn_rows = int(N * VOL_FRAC_SN)
grid_par[:num_sn_rows, :] = 0
rho_par, V_par = solve_resistor_network_corrected(grid_par, RHO_SN, RHO_BI, R_INT_VAL)

# 2. Series (Worst)
grid_ser = np.ones((N, N))
num_sn_cols = int(N * VOL_FRAC_SN)
grid_ser[:, :num_sn_cols] = 0
rho_ser, V_ser = solve_resistor_network_corrected(grid_ser, RHO_SN, RHO_BI, R_INT_VAL)

# 3. Random (Realistic Mix)
grid_rand = np.random.choice([0, 1], size=(N, N), p=[VOL_FRAC_SN, 1-VOL_FRAC_SN])
rho_rand, V_rand = solve_resistor_network_corrected(grid_rand, RHO_SN, RHO_BI, R_INT_VAL)

# --- RESULTS ---
print("\n" + "="*50)
print(f"CORRECTED BOUNDS (Sn={RHO_SN}, Bi={RHO_BI})")
print("="*50)
print(f"1. Parallel Limit (Ideal Highway):   {rho_par:.1f} uOhm-cm")
print(f"2. Random Limit (Disconnected):      {rho_rand:.1f} uOhm-cm")
print(f"3. Series Limit (Blocked Wall):      {rho_ser:.1f} uOhm-cm")
print("-" * 50)
print(f"YOUR SIMULATION (Code 6 Growth):     ~51.2 uOhm-cm")
print(f"CONCLUSION: Growth improves conductivity by connecting the random islands.")
print("="*50)

# Plots
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(grid_par, cmap='gray_r'); ax[0].set_title(f"Parallel\nRho={rho_par:.1f}")
ax[1].imshow(grid_rand, cmap='gray_r'); ax[1].set_title(f"Random\nRho={rho_rand:.1f}")
ax[2].imshow(grid_ser, cmap='gray_r'); ax[2].set_title(f"Series\nRho={rho_ser:.1f}")
plt.show()

ModuleNotFoundError: No module named 'numpy'

# 2. Failed Grain Model


In [ ]:
# --- NEW FUNCTION: Grain Structure Generator ---
def generate_grainy_grid(N, block_size, vol_frac):
    """
    Generates a grid made of distinct blocks (grains).
    Each block is internally either 'Parallel' or 'Series'.
    """
    grid = np.zeros((N, N))

    # Calculate how many blocks fit
    num_blocks = N // block_size

    for r in range(num_blocks):
        for c in range(num_blocks):
            # Define the coordinates of this block
            y_start, y_end = r * block_size, (r + 1) * block_size
            x_start, x_end = c * block_size, (c + 1) * block_size

            # 1. Flip a coin: Is this block Horizontal or Vertical?
            orientation = np.random.choice(['horizontal', 'vertical'])

            # 2. Fill the block based on orientation
            # We create a mini-grid for just this block
            mini_block = np.ones((block_size, block_size)) # Start as Bi (1)

            if orientation == 'horizontal':
                # Stripes running Left-Right (Parallel-like)
                num_sn_rows = int(block_size * vol_frac)
                mini_block[:num_sn_rows, :] = 0 # Top part is Sn

            else: # vertical
                # Stripes running Up-Down (Series-like)
                num_sn_cols = int(block_size * vol_frac)
                mini_block[:, :num_sn_cols] = 0 # Left part is Sn

            # 3. Paste the mini-block into the main grid
            grid[y_start:y_end, x_start:x_end] = mini_block

    return grid

# --- MAIN EXECUTION (Updated) ---

print("Running Grain Simulation... (Generating 3 random samples)")

# Constants
BLOCK_SIZE = 15  # Each grain is 15x15 pixels
N = 90           # Larger total grid (90x90) to fit enough blocks

# Generate 3 different random "Grainy" samples to see the variation
grids = []
results = []

for i in range(3):
    # Generate the grid using your "Block Idea"
    g = generate_grainy_grid(N, BLOCK_SIZE, VOL_FRAC_SN)
    grids.append(g)

    # Run the solver
    rho, _ = solve_resistor_network_fast(g, RHO_SN, RHO_BI, R_INT_VAL)
    results.append(rho)
    print(f"Sample {i+1}: Resistivity = {rho:.2f} uOhm-cm")

# --- PLOTTING ---
fig, ax = plt.subplots(1, 3, figsize=(18, 6))

for i in range(3):
    ax[i].imshow(grids[i], cmap='gray_r', interpolation='nearest')
    ax[i].set_title(f"Grainy Sample {i+1}\nRho = {results[i]:.2f}")
    # Draw grid lines to show where the blocks are
    ax[i].set_xticks(np.arange(0, N, BLOCK_SIZE)-0.5, minor=True)
    ax[i].set_yticks(np.arange(0, N, BLOCK_SIZE)-0.5, minor=True)
    ax[i].grid(which='minor', color='red', linestyle='-', linewidth=1)

plt.suptitle(f"The '{BLOCK_SIZE}x{BLOCK_SIZE} Block' Model (Red lines show block boundaries)", fontsize=16)
plt.tight_layout()
plt.show()

# 3. Slow Spinodal Model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

# --- ADVANCED CONFIGURATION ---
N = 300                 # Large Grid (300x300 pixels)
BLOCK_SIZE = 15         # Grain size (15x15 pixels)
VOL_FRAC_SN = 0.42      # Exact Eutectic Sn fraction (by volume)
RHO_SN = 11.5           # Tin resistivity
RHO_BI = 129.0          # Bismuth resistivity
R_INT_VAL = 20.0        # Interface penalty
NUM_SIMULATIONS = 10    # Monte Carlo runs (Increase this for better stats)
ITERATIONS = 8000       # Solver precision (More iterations = more accurate)

# --- HELPER FUNCTIONS ---

def generate_grainy_grid(N, block_size, vol_frac):
    """Generates a large grid of randomized parallel/series blocks."""
    grid = np.zeros((N, N))
    num_blocks = N // block_size

    for r in range(num_blocks):
        for c in range(num_blocks):
            y_start, y_end = r * block_size, (r + 1) * block_size
            x_start, x_end = c * block_size, (c + 1) * block_size

            # Random Orientation: 0=Horizontal(Parallel), 1=Vertical(Series)
            orientation = np.random.choice(['h', 'v'])
            mini_block = np.ones((block_size, block_size))

            if orientation == 'h':
                # Sn stripes run Left-Right
                num_sn_rows = int(block_size * vol_frac)
                mini_block[:num_sn_rows, :] = 0
            else:
                # Sn stripes run Up-Down
                num_sn_cols = int(block_size * vol_frac)
                mini_block[:, :num_sn_cols] = 0

            grid[y_start:y_end, x_start:x_end] = mini_block
    return grid

def solve_field(grid, rho_A, rho_B, r_interface, max_iter):
    """Solves Voltage field and calculates Current Density."""
    ny, nx = grid.shape
    V = np.zeros((ny, nx))
    V[:, 0] = 1.0; V[:, -1] = 0.0 # Boundary Conditions

    # Material Resistivity Map
    rho_map = np.where(grid == 0, rho_A, rho_B)

    # Pre-calculate conductance arrays to speed up the loop
    # Right Neighbors
    R_right = (rho_map[:, :-1] + rho_map[:, 1:]) / 2
    R_right[grid[:, :-1] != grid[:, 1:]] += r_interface
    G_right = 1.0 / R_right

    # Down Neighbors
    R_down = (rho_map[:-1, :] + rho_map[1:, :]) / 2
    R_down[grid[:-1, :] != grid[1:, :]] += r_interface
    G_down = 1.0 / R_down

    # --- MAIN SOLVER LOOP (Finite Difference) ---
    # Using a checkerboard update (Red-Black SOR) could be faster,
    # but standard relaxation is safer for "ragged" matrices like this.
    for k in range(max_iter):
        # Calculate weighted average from neighbors
        term_right = V[1:-1, 2:] * G_right[1:-1, 1:]
        term_left  = V[1:-1, :-2] * G_right[1:-1, :-1]
        term_down  = V[2:, 1:-1] * G_down[1:, 1:-1]
        term_up    = V[:-2, 1:-1] * G_down[:-1, 1:-1]

        G_sum = (G_right[1:-1, 1:] + G_right[1:-1, :-1] +
                 G_down[1:, 1:-1] + G_down[:-1, 1:-1])

        V[1:-1, 1:-1] = (term_right + term_left + term_down + term_up) / G_sum

    # --- CALCULATE PHYSICS OUTPUTS ---
    # 1. Total Current (flux through right boundary)
    currents = (V[:, -2] - V[:, -1]) / R_right[:, -1]
    total_current = np.sum(currents)

    # 2. Effective Resistivity
    rho_eff = (1.0 / total_current) * (ny / nx) if total_current > 0 else 1e9

    # 3. Current Density Magnitude (J) map for visualization
    # J = E / Rho.   E = grad(V).
    # This is an approximation for visualization
    Jy = np.zeros_like(V); Jx = np.zeros_like(V)
    Jx[:, :-1] = (V[:, :-1] - V[:, 1:]) / R_right # Forward diff
    Jy[:-1, :] = (V[:-1, :] - V[1:, :]) / R_down
    J_mag = np.sqrt(Jx**2 + Jy**2)

    return rho_eff, V, J_mag

# --- EXECUTION ENGINE ---
print(f"Starting Monte Carlo Simulation ({NUM_SIMULATIONS} runs)...")
print(f"Grid: {N}x{N} pixels | Grains: ~{int((N/BLOCK_SIZE)**2)}")
print("-" * 50)

results = []
best_run = None
worst_run = None
min_rho = float('inf')
max_rho = float('-inf')

start_time = time.time()

for i in range(NUM_SIMULATIONS):
    # Generate new random structure
    g = generate_grainy_grid(N, BLOCK_SIZE, VOL_FRAC_SN)

    # Solve
    rho, V, J = solve_field(g, RHO_SN, RHO_BI, R_INT_VAL, ITERATIONS)
    results.append(rho)

    # Track Best/Worst for plotting
    if rho < min_rho:
        min_rho = rho
        best_run = (g, V, J)
    if rho > max_rho:
        max_rho = rho
        worst_run = (g, V, J)

    print(f"Run {i+1}/{NUM_SIMULATIONS}: Rho = {rho:.2f} uOhm-cm")

total_time = time.time() - start_time
mean_rho = np.mean(results)
std_rho = np.std(results)

# --- REPORTING ---
print("-" * 50)
print(f"SIMULATION COMPLETE in {total_time:.1f} seconds")
print(f"Average Resistivity: {mean_rho:.2f} +/- {std_rho:.2f} uOhm-cm")
print("-" * 50)

# --- VISUALIZATION ---
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Histogram of Results
axes[0,0].hist(results, bins=10, color='skyblue', edgecolor='black')
axes[0,0].axvline(mean_rho, color='red', linestyle='dashed', linewidth=2)
axes[0,0].set_title(f"Distribution of Resistivity\nMean: {mean_rho:.1f} (Red Line)")
axes[0,0].set_xlabel("Resistivity (uOhm-cm)")
axes[0,0].set_ylabel("Frequency")

# 2. Best Case Microstructure (Lowest R)
axes[0,1].imshow(best_run[0], cmap='gray_r')
axes[0,1].set_title(f"Best Case Structure (Rho={min_rho:.1f})\nNotice the connected black paths")
axes[0,1].axis('off')

# 3. Current Density Map (The "Heatmap")
# We use log scale because current density varies wildly
im3 = axes[1,0].imshow(np.log1p(best_run[2]), cmap='inferno')
axes[1,0].set_title("Current Density Map (Best Case)\nBright = High Current Flow")
axes[1,0].axis('off')
plt.colorbar(im3, ax=axes[1,0], fraction=0.046, pad=0.04)

# 4. Worst Case Microstructure
axes[1,1].imshow(worst_run[0], cmap='gray_r')
axes[1,1].set_title(f"Worst Case Structure (Rho={max_rho:.1f})\nNotice the white (Bi) blockage")
axes[1,1].axis('off')

plt.tight_layout()
plt.show()

# 4. Fast Matrix Model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.ndimage as ndimage
from scipy.sparse import lil_matrix, csc_matrix
from scipy.sparse.linalg import spsolve
import time

# --- CONFIGURATION ---
N = 300                 # Grid size (300x300)
# CRITICAL FIX: We use 0.58 in 2D to mimic the connectivity of 0.45 in 3D.
# This ensures the "Sn" phase actually touches from left to right.
VOL_FRAC_GENERATION = 0.58

RHO_SN = 11.5           # Tin (Conductive)
RHO_BI = 129.0          # Bismuth (Resistive)
R_INT_VAL = 5.0         # Interface Penalty
GRAIN_SCALE = 8.0       # Structure size

def generate_connected_microstructure(N, scale, vol_frac):
    """
    Generates a spinodal-like structure.
    Uses Gaussian fields to create smooth, connected 'veins'.
    """
    noise = np.random.randn(N, N)
    smooth_noise = ndimage.gaussian_filter(noise, sigma=scale)
    threshold = np.percentile(smooth_noise, vol_frac * 100)

    # 0 = Sn (Conductive), 1 = Bi (Resistive)
    grid = np.ones((N, N))
    grid[smooth_noise < threshold] = 0
    return grid

def solve_sparse_matrix(grid, rho_A, rho_B, r_interface):
    """
    The 'Pro' Solver: Uses Matrix Algebra (Ax=b) to solve instantly.
    """
    ny, nx = grid.shape
    num_pixels = ny * nx

    # 1. Setup Resistivity Map
    rho_map = np.where(grid == 0, rho_A, rho_B)

    # 2. Build the Sparse Matrix (A) and RHS vector (b)
    # Equation: Sum(Conductance * (V_neighbor - V_center)) = 0
    A = lil_matrix((num_pixels, num_pixels))
    b = np.zeros(num_pixels)

    # Helper to convert (row, col) to flattened index
    def idx(r, c): return r * nx + c

    # --- FILL MATRIX (Vectorized for speed) ---
    # This part is tricky to read but very fast.
    # We define conductances between every pair of pixels.

    # Horizontal Conductances (Right)
    # R between (r,c) and (r,c+1)
    r_link_h = (rho_map[:, :-1] + rho_map[:, 1:]) / 2.0
    # Add interface penalty
    mask_h = (grid[:, :-1] != grid[:, 1:])
    r_link_h[mask_h] += r_interface
    g_h = 1.0 / r_link_h # Conductance

    # Vertical Conductances (Down)
    # R between (r,c) and (r+1,c)
    r_link_v = (rho_map[:-1, :] + rho_map[1:, :]) / 2.0
    mask_v = (grid[:-1, :] != grid[1:, :])
    r_link_v[mask_v] += r_interface
    g_v = 1.0 / r_link_v

    # We populate the matrix.
    # For a pixel k, A[k,k] = sum(conductances to neighbors)
    # A[k, neighbor] = -conductance

    # This is a simplified construction for readability in Colab:
    # (In a full library we'd build the COO arrays directly)

    for r in range(ny):
        for c in range(nx):
            k = idx(r, c)

            # Boundary Conditions
            if c == 0: # Left Wall (Voltage = 1.0)
                A[k, k] = 1.0
                b[k] = 1.0
                continue
            elif c == nx - 1: # Right Wall (Voltage = 0.0)
                A[k, k] = 1.0
                b[k] = 0.0
                continue

            # Interior Nodes (Kirchhoff's Current Law)
            g_sum = 0.0

            # Left Neighbor
            if c > 0:
                g = g_h[r, c-1]
                A[k, idx(r, c-1)] = -g
                g_sum += g

            # Right Neighbor
            if c < nx - 1:
                g = g_h[r, c]
                A[k, idx(r, c+1)] = -g
                g_sum += g

            # Up Neighbor
            if r > 0:
                g = g_v[r-1, c]
                A[k, idx(r-1, c)] = -g
                g_sum += g

            # Down Neighbor
            if r < ny - 1:
                g = g_v[r, c]
                A[k, idx(r+1, c)] = -g
                g_sum += g

            A[k, k] = g_sum

    # 3. Solve Ax = b
    # Convert to CSC format for fast solving
    A_csc = A.tocsc()
    V_flat = spsolve(A_csc, b)
    V = V_flat.reshape((ny, nx))

    # 4. Calculate Current
    # Current leaving column -2 into column -1
    currents = (V[:, -2] - V[:, -1]) / r_link_h[:, -1]
    total_current = np.sum(currents)

    rho_eff = (1.0 / total_current) * (ny / nx)
    return rho_eff, grid, V

# --- MAIN RUN ---
print("Running Fast Matrix Solver...")
start_time = time.time()

# Generate Grid with "3D Connectivity" fix
grid = generate_connected_microstructure(N, GRAIN_SCALE, VOL_FRAC_GENERATION)

# Solve
rho, grid_out, V = solve_sparse_matrix(grid, RHO_SN, RHO_BI, R_INT_VAL)

elapsed = time.time() - start_time
print(f"Done in {elapsed:.2f} seconds.")
print("="*40)
print(f"CALCULATED RESISTIVITY: {rho:.2f} uOhm-cm")
print("="*40)

# --- PLOT ---
fig, ax = plt.subplots(1, 2, figsize=(12, 6))

# Microstructure
ax[0].imshow(grid, cmap='gray_r')
ax[0].set_title(f"Interconnected Phase Model\n(Black=Sn, White=Bi)")
ax[0].axis('off')

# Voltage Map
im = ax[1].imshow(V, cmap='plasma')
ax[1].set_title("Voltage Drop Map\n(Smooth gradient = Good connectivity)")
ax[1].axis('off')
plt.colorbar(im, ax=ax[1])

plt.tight_layout()
plt.show()

# 5. High-Fidelity Intricate Model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.ndimage as ndimage
from scipy.sparse import lil_matrix, csc_matrix
from scipy.sparse.linalg import spsolve
import time

# # --- CONFIGURATION ---
# N = 400                 # Higher Resolution (400x400) for finer detail
# VOL_FRAC_GENERATION = 0.58

# RHO_SN = 11.5
# RHO_BI = 129.0
# R_INT_VAL = 5.0

# --- CALIBRATED PHYSICS PARAMETERS ---
# 1. Solid Solution Correction: Sn is not pure; it has dissolved Bi.
RHO_SN = 18.0           # Increased from 11.5 to 18.0 (Solid Solution Effect)
RHO_BI = 129.0          # Stays the same

# 2. Interface Scattering Dominance: The "Border Tax"
# Suk et al. (1994) says interfaces are the primary resistive barrier.
# We increase this until the result matches the 50.5 target.
R_INT_VAL = 45.0        # Increased from 5.0 to 45.0

# 3. Connectivity Balance
# Keep the volume fraction high enough to connect (3D correction),
# but let the parameters above do the work of adding resistance.
VOL_FRAC_GENERATION = 0.58

# --- NEW: DUAL SCALE PARAMETERS ---
SCALE_PRIMARY = 10.0    # Big "Trunk" structures
SCALE_SECONDARY = 2.0   # Fine "Branch" details
DETAIL_WEIGHT = 0.4     # How strong the fine details are (0.0 to 1.0)

def generate_intricate_microstructure(N, scale_main, scale_detail, weight, vol_frac):
    """
    Generates a complex, multi-scale microstructure.
    Combines coarse 'primary' grains with fine 'secondary' roughness.
    """
    # 1. Primary Structure (The Big Grains)
    noise_main = np.random.randn(N, N)
    field_main = ndimage.gaussian_filter(noise_main, sigma=scale_main)

    # 2. Secondary Structure (The Intricate Details)
    noise_detail = np.random.randn(N, N)
    field_detail = ndimage.gaussian_filter(noise_detail, sigma=scale_detail)

    # 3. Combine them (Weighted Average)
    # This creates 'rough' blobs instead of smooth ones
    combined_field = (1.0 - weight) * field_main + weight * field_detail

    # 4. Threshold to get phases
    threshold = np.percentile(combined_field, vol_frac * 100)

    grid = np.ones((N, N))
    grid[combined_field < threshold] = 0 # 0 = Sn
    return grid

def solve_sparse_matrix(grid, rho_A, rho_B, r_interface):
    # (Same Optimized Solver as before)
    ny, nx = grid.shape
    num_pixels = ny * nx
    rho_map = np.where(grid == 0, rho_A, rho_B)

    A = lil_matrix((num_pixels, num_pixels))
    b = np.zeros(num_pixels)
    def idx(r, c): return r * nx + c

    # Pre-calculate link conductances
    # Horizontal
    r_h = (rho_map[:, :-1] + rho_map[:, 1:]) / 2.0
    r_h[grid[:, :-1] != grid[:, 1:]] += r_interface
    g_h = 1.0 / r_h

    # Vertical
    r_v = (rho_map[:-1, :] + rho_map[1:, :]) / 2.0
    r_v[grid[:-1, :] != grid[1:, :]] += r_interface
    g_v = 1.0 / r_v

    # Fill Matrix (Simplified loop for clarity)
    for r in range(ny):
        for c in range(nx):
            k = idx(r, c)
            if c == 0:
                A[k, k] = 1.0; b[k] = 1.0
                continue
            elif c == nx - 1:
                A[k, k] = 1.0; b[k] = 0.0
                continue

            g_sum = 0.0
            if c > 0:       g = g_h[r, c-1]; A[k, idx(r, c-1)] = -g; g_sum += g
            if c < nx - 1:  g = g_h[r, c];   A[k, idx(r, c+1)] = -g; g_sum += g
            if r > 0:       g = g_v[r-1, c]; A[k, idx(r-1, c)] = -g; g_sum += g
            if r < ny - 1:  g = g_v[r, c];   A[k, idx(r+1, c)] = -g; g_sum += g
            A[k, k] = g_sum

    V_flat = spsolve(A.tocsc(), b)
    V = V_flat.reshape((ny, nx))

    # Current Calc
    currents = (V[:, -2] - V[:, -1]) / r_h[:, -1]
    total_current = np.sum(currents)
    rho_eff = (1.0 / total_current) * (ny / nx)
    return rho_eff, grid, V

# --- MAIN RUN ---
print("Generating Intricate Dual-Scale Microstructure...")
start = time.time()

# Generate with Dual-Scale
grid = generate_intricate_microstructure(N, SCALE_PRIMARY, SCALE_SECONDARY, DETAIL_WEIGHT, VOL_FRAC_GENERATION)

# Solve
rho, grid_out, V = solve_sparse_matrix(grid, RHO_SN, RHO_BI, R_INT_VAL)

elapsed = time.time() - start
print(f"Solved 400x400 grid in {elapsed:.2f} seconds.")
print(f"Resistivity: {rho:.2f} uOhm-cm")

# --- PLOTTING ---
fig, ax = plt.subplots(1, 2, figsize=(14, 7), dpi=100) # Higher DPI for detail

ax[0].imshow(grid, cmap='gray_r', interpolation='nearest')
ax[0].set_title(f"Intricate Microstructure\n(Notice the rough, jagged edges)")
ax[0].axis('off')

im = ax[1].imshow(V, cmap='magma')
ax[1].set_title("Voltage Map")
ax[1].axis('off')
plt.colorbar(im, ax=ax[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

# 6. Thermodynamic Kinetic Monte Carlo (KMC) Model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import lil_matrix, csc_matrix
from scipy.sparse.linalg import spsolve
import time
import numba # We need this to make the slow simulation fast enough to finish today

# --- CALIBRATED PHYSICS PARAMETERS ---
N = 400                  # Grid Size (400x400 atoms)
VOL_FRAC = 0.58          # Adjusted Volume Fraction for 3D Connectivity
MCS_STEPS = 500          # "Monte Carlo Steps" (1 MCS = N*N attempts)
                         # Total atomic swaps = 400*400 * 500 = 80 Million swaps!
TEMPERATURE = 0.5        # Simulation Temperature (controls roughness)

# Resistivity Constants (Calibrated)
RHO_SN = 18.0            # Dirty Tin (Solid Solution)
RHO_BI = 129.0           # Bismuth
R_INT_VAL = 45.0         # Interface Penalty

# --- PART 1: KINETIC MONTE CARLO (The "Growth" Engine) ---
# We use Numba (Just-In-Time Compiler) to speed up Python loops by 100x
# If Numba is not installed, this will be slow. In Colab, it usually works.

@numba.jit(nopython=True)
def count_neighbors(grid, r, c, val, N):
    """Counts how many neighbors match the value 'val'."""
    count = 0
    # Periodic Boundary Conditions (Wrap around like Pac-Man)
    # This simulates an 'infinite' bulk material
    if grid[(r+1)%N, c] == val: count += 1
    if grid[(r-1)%N, c] == val: count += 1
    if grid[r, (c+1)%N] == val: count += 1
    if grid[r, (c-1)%N] == val: count += 1
    return count

@numba.jit(nopython=True)
def run_monte_carlo(grid, steps, temp, N):
    """
    Simulates grain growth via surface energy minimization (Metropolis Algorithm).
    """
    total_swaps = N * N

    for step in range(steps):
        # In one 'Step', we try to swap every atom once on average
        for _ in range(total_swaps):
            # 1. Pick two random random locations
            r1, c1 = np.random.randint(0, N), np.random.randint(0, N)
            r2, c2 = np.random.randint(0, N), np.random.randint(0, N)

            spin1 = grid[r1, c1]
            spin2 = grid[r2, c2]

            # If they are already same, swapping does nothing
            if spin1 == spin2: continue

            # 2. Calculate Energy BEFORE swap
            # Energy = number of 'unhappy' neighbors (different spins)
            # We want to MAXIMIZE 'happy' neighbors (same spin)
            like_neighbors_1_pre = count_neighbors(grid, r1, c1, spin1, N)
            like_neighbors_2_pre = count_neighbors(grid, r2, c2, spin2, N)

            # 3. Calculate Energy AFTER swap
            # (Hypothetically swap them)
            like_neighbors_1_post = count_neighbors(grid, r1, c1, spin2, N) # At loc 1, we would have spin2
            like_neighbors_2_post = count_neighbors(grid, r2, c2, spin1, N) # At loc 2, we would have spin1

            # Change in "Happiness" (Negative Energy Change)
            # We want to increase like-neighbors.
            gain = (like_neighbors_1_post + like_neighbors_2_post) - \
                   (like_neighbors_1_pre + like_neighbors_2_pre)

            # 4. Metropolis Criterion
            # If gain > 0 (Lower Energy), always swap.
            # If gain <= 0 (Higher Energy), swap with probability exp(gain/T).
            if gain > 0 or np.random.random() < np.exp(gain / temp):
                grid[r1, c1] = spin2
                grid[r2, c2] = spin1

    return grid

# --- PART 2: RESISTIVITY SOLVER (The "Probe" Engine) ---

def solve_resistivity(grid, rho_A, rho_B, r_interface):
    ny, nx = grid.shape
    num_pixels = ny * nx
    rho_map = np.where(grid == 0, rho_A, rho_B)

    A = lil_matrix((num_pixels, num_pixels))
    b = np.zeros(num_pixels)
    def idx(r, c): return r * nx + c

    # Pre-calculate conductances
    # Horizontal
    r_h = (rho_map[:, :-1] + rho_map[:, 1:]) / 2.0
    r_h[grid[:, :-1] != grid[:, 1:]] += r_interface
    g_h = 1.0 / r_h

    # Vertical
    r_v = (rho_map[:-1, :] + rho_map[1:, :]) / 2.0
    r_v[grid[:-1, :] != grid[1:, :]] += r_interface
    g_v = 1.0 / r_v

    print(f"Building Matrix ({num_pixels} equations)...")
    for r in range(ny):
        for c in range(nx):
            k = idx(r, c)
            if c == 0:
                A[k, k] = 1.0; b[k] = 1.0
                continue
            elif c == nx - 1:
                A[k, k] = 1.0; b[k] = 0.0
                continue

            g_sum = 0.0
            if c > 0:       g = g_h[r, c-1]; A[k, idx(r, c-1)] = -g; g_sum += g
            if c < nx - 1:  g = g_h[r, c];   A[k, idx(r, c+1)] = -g; g_sum += g
            if r > 0:       g = g_v[r-1, c]; A[k, idx(r-1, c)] = -g; g_sum += g
            if r < ny - 1:  g = g_v[r, c];   A[k, idx(r+1, c)] = -g; g_sum += g
            A[k, k] = g_sum

    print("Solving Sparse System...")
    V_flat = spsolve(A.tocsc(), b)
    V = V_flat.reshape((ny, nx))

    currents = (V[:, -2] - V[:, -1]) / r_h[:, -1]
    total_current = np.sum(currents)
    rho_eff = (1.0 / total_current) * (ny / nx)
    return rho_eff, V

# --- MAIN EXECUTION ---
print(f"--- STARTING SIMULATION ---")
print(f"System Size: {N}x{N} atoms")
print(f"Simulation Steps: {MCS_STEPS} Monte Carlo Sweeps (This will take time...)")

start_time = time.time()

# 1. Initialize Random Liquid
# 0 = Sn, 1 = Bi
grid = np.random.choice([0, 1], size=(N, N), p=[VOL_FRAC, 1-VOL_FRAC])

# 2. Run Grain Growth (Physics!)
print("Growing Microstructure...")
final_grid = run_monte_carlo(grid, MCS_STEPS, TEMPERATURE, N)
growth_time = time.time() - start_time
print(f"Growth Completed in {growth_time:.1f} seconds.")

# 3. Measure Resistivity
print("Measuring Electrical Properties...")
rho, V = solve_resistivity(final_grid, RHO_SN, RHO_BI, R_INT_VAL)

total_time = time.time() - start_time
print("="*50)
print(f"TOTAL SIMULATION TIME: {total_time/60:.1f} Minutes")
print(f"FINAL RESISTIVITY: {rho:.2f} uOhm-cm")
print("="*50)

# --- VISUALIZATION ---
fig, ax = plt.subplots(1, 2, figsize=(16, 8))

ax[0].imshow(final_grid, cmap='gray_r')
ax[0].set_title(f"Thermodynamically Grown Microstructure\n(Kinetic Monte Carlo, T={TEMPERATURE})")
ax[0].axis('off')

im = ax[1].imshow(V, cmap='magma')
ax[1].set_title(f"Voltage Field\nRho = {rho:.2f} uOhm-cm")
ax[1].axis('off')
plt.colorbar(im, ax=ax[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

# 6-2. Updated for volume fraction

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import csc_matrix
from scipy.sparse.linalg import spsolve
import time
import numba

# --- 1. CONFIGURATION (The "Hero Run" Settings) ---
N = 1000                 # Grid Size: 1000x1000 (1 Million Atoms)
                         # WARNING: This requires ~8GB RAM for the solver.

# Corrected Volume Fraction (Density Adjusted)
# Sn-58wt%Bi -> Density Sn=7.3, Bi=9.8 -> Vol% Sn approx 49%
VOL_FRAC_SN = 0.49

MCS_STEPS = 400          # Monte Carlo Steps (Slightly reduced for speed at N=1000)
TEMPERATURE = 0.5        # Simulation Temperature (Grain Roughness)

# Resistivity Constants (Calibrated to Room Temp 20C)
RHO_SN = 18.0            # Tin (Solid Solution Hardened)
RHO_BI = 129.0           # Bismuth
R_INT_VAL = 45.0         # Interface Scattering Penalty

# --- 2. KINETIC MONTE CARLO (Grain Growth Engine) ---
@numba.jit(nopython=True)
def count_neighbors_fast(grid, r, c, val, size):
    """
    Optimized neighbor counting with Periodic Boundary Conditions.
    """
    count = 0
    # Pre-calculate indices to avoid modulo in every check if possible,
    # but modulo is safest for periodic boundaries.
    r_up = (r - 1) % size
    r_down = (r + 1) % size
    c_left = (c - 1) % size
    c_right = (c + 1) % size

    if grid[r_up, c] == val: count += 1
    if grid[r_down, c] == val: count += 1
    if grid[r, c_left] == val: count += 1
    if grid[r, c_right] == val: count += 1
    return count

@numba.jit(nopython=True)
def run_monte_carlo_fast(grid, steps, temp, size):
    """
    Metropolis-Hastings Algorithm for Grain Growth (Potts Model).
    """
    total_sites = size * size
    # Attempt 1 swap per site per step (on average)
    swaps_per_step = total_sites

    for step in range(steps):
        for _ in range(swaps_per_step):
            # 1. Pick two random sites
            r1 = np.random.randint(0, size)
            c1 = np.random.randint(0, size)
            r2 = np.random.randint(0, size)
            c2 = np.random.randint(0, size)

            spin1 = grid[r1, c1]
            spin2 = grid[r2, c2]

            if spin1 == spin2: continue

            # 2. Calculate Energy Change (Delta E)
            # We want to maximize 'like' neighbors (Happiness)

            # Initial Happiness
            h1_pre = count_neighbors_fast(grid, r1, c1, spin1, size)
            h2_pre = count_neighbors_fast(grid, r2, c2, spin2, size)

            # Hypothetical Post-Swap Happiness
            # (Note: Strictly we should check if r1/r2 are neighbors,
            # but for N=1000 the error probability is <0.0001%, negligible for speed)
            h1_post = count_neighbors_fast(grid, r1, c1, spin2, size)
            h2_post = count_neighbors_fast(grid, r2, c2, spin1, size)

            # Change in Happiness (Energy is negative happiness)
            delta_happiness = (h1_post + h2_post) - (h1_pre + h2_pre)

            # 3. Metropolis Acceptance
            # If we gain happiness, ALWAYS swap.
            # If we lose happiness, swap with probability exp(delta / T)
            if delta_happiness > 0:
                grid[r1, c1] = spin2
                grid[r2, c2] = spin1
            elif np.random.random() < np.exp(delta_happiness / temp):
                grid[r1, c1] = spin2
                grid[r2, c2] = spin1

    return grid

# --- 3. FINITE DIFFERENCE SOLVER (Bakker Method) ---
def solve_resistivity_large_scale(grid, rho_A, rho_B, r_interface):
    """
    Constructs and solves the sparse linear system for 1 Million+ unknowns.
    """
    ny, nx = grid.shape
    num_pixels = ny * nx

    print(f"  > Mapping Resistivity Field...")
    rho_map = np.where(grid == 0, rho_A, rho_B).astype(np.float64)

    # Helper indexer
    def idx(r, c): return r * nx + c

    print(f"  > Building Sparse Matrix ({num_pixels} Equations)...")

    # Lists for COO format construction (efficient for memory)
    rows = []
    cols = []
    data = []
    rhs = np.zeros(num_pixels)

    # We loop via Numba or Vectorization would be faster, but for clarity
    # and memory safety on huge grids, we use a robust Python loop generator approach
    # or simple pre-calculation.
    # To speed this up for N=1000, we will assume standard neighbor logic without loop:

    # -- VECTORIZED GRAPH CONSTRUCTION (Speedup for N=1000) --
    # Indices
    Y, X = np.indices((ny, nx))
    node_indices = Y * nx + X

    # 1. Horizontal Links (Right Neighbors)
    # Mask for all columns except last
    mask_h = (X < nx - 1)

    r_curr = rho_map[mask_h]
    r_next = rho_map[:, 1:][mask_h[:, :-1]] # Shifted right

    # Base Resistance
    r_link_h = (r_curr + r_next) / 2.0

    # Interface Penalty
    diff_mask_h = (grid[mask_h] != grid[:, 1:][mask_h[:, :-1]])
    r_link_h[diff_mask_h] += r_interface

    g_h = 1.0 / r_link_h

    # Add to Sparse Matrix
    # Current node gets +g, Right neighbor gets -g
    # We do this by accumulating conductance 'g_sum' for the diagonal later

    # 2. Vertical Links (Down Neighbors)
    mask_v = (Y < ny - 1)
    r_curr_v = rho_map[mask_v]
    r_next_v = rho_map[1:, :][mask_v[:-1, :]] # Shifted down

    r_link_v = (r_curr_v + r_next_v) / 2.0
    diff_mask_v = (grid[mask_v] != grid[1:, :][mask_v[:-1, :]])
    r_link_v[diff_mask_v] += r_interface

    g_v = 1.0 / r_link_v

    # -- SYSTEM ASSEMBLY --
    # This part is tricky to fully vectorize without complex indexing,
    # so we stick to the reliable list build but strictly efficient.
    # Actually, let's revert to the reliable loop for safety,
    # but print progress so you know it hasn't crashed.

    print("  > Assembling Connectivity (This may take 30-60s)...")

    # Re-using the logic from before but careful with memory
    # We will build simple arrays

    # Nodes
    ids = np.arange(num_pixels).reshape(ny, nx)

    # Right Neighbors
    # Nodes (r, c) connects to (r, c+1)
    # Exclude last col
    u = ids[:, :-1].flatten()
    v = ids[:, 1:].flatten()
    g = g_h.flatten()

    # Append +g to diagonal of u, -g to v, etc.
    # We need a robust way:
    # Interaction u-v adds:
    # Eq u:  G*(Vu - Vv) ... -> +G to u, -G to v
    # Eq v:  G*(Vv - Vu) ... -> +G to v, -G to u

    # We collect all triplets
    all_rows = []
    all_cols = []
    all_vals = []

    # Horizontal couplings
    all_rows.append(u); all_cols.append(u); all_vals.append(g)  # u-u
    all_rows.append(u); all_cols.append(v); all_vals.append(-g) # u-v
    all_rows.append(v); all_cols.append(v); all_vals.append(g)  # v-v
    all_rows.append(v); all_cols.append(u); all_vals.append(-g) # v-u

    # Vertical couplings
    u_v = ids[:-1, :].flatten()
    v_v = ids[1:, :].flatten()
    g_v_flat = g_v.flatten()

    all_rows.append(u_v); all_cols.append(u_v); all_vals.append(g_v_flat)
    all_rows.append(u_v); all_cols.append(v_v); all_vals.append(-g_v_flat)
    all_rows.append(v_v); all_cols.append(v_v); all_vals.append(g_v_flat)
    all_rows.append(v_v); all_cols.append(u_v); all_vals.append(-g_v_flat)

    # Concatenate
    rows_arr = np.concatenate(all_rows)
    cols_arr = np.concatenate(all_cols)
    vals_arr = np.concatenate(all_vals)

    # Create Matrix
    A = csc_matrix((vals_arr, (rows_arr, cols_arr)), shape=(num_pixels, num_pixels))

    # -- BOUNDARY CONDITIONS (Source/Drain) --
    # We need to enforce V=1 at left, V=0 at right.
    # The 'hard' way in sparse matrices is replacing rows.
    # We will use the "Large Penalty" method or direct replacement.
    # Direct replacement is cleaner for the solver.

    # Left Nodes (col 0)
    left_nodes = ids[:, 0]
    # Right Nodes (col last)
    right_nodes = ids[:, -1]

    # We need to zero out the rows for these nodes and put a 1.0 on diagonal
    # This is slow in CSC. LIL is better for structure changes, but CSC is better for solving.
    # Strategy: Construct A_modified directly.

    # Actually, simpler approach for 1M atoms:
    # Use "Penalty Method". Add a HUGE number to the diagonal and HUGE*Value to RHS.
    # A[k,k] += 1e12
    # b[k] += 1e12 * 1.0

    PENALTY = 1e15

    # Convert to LIL for BC application (takes a moment but worth it)
    # A = A.tolil()
    # Optimization: Don't convert. Add penalty to existing diagonal.

    diag_indices_left = left_nodes # index is (k,k)
    diag_indices_right = right_nodes

    # We create a correction matrix to add to A
    row_bc = np.concatenate([left_nodes, right_nodes])
    col_bc = np.concatenate([left_nodes, right_nodes])
    val_bc = np.ones(len(row_bc)) * PENALTY

    A_bc = csc_matrix((val_bc, (row_bc, col_bc)), shape=(num_pixels, num_pixels))
    A_final = A + A_bc

    # RHS
    b = np.zeros(num_pixels)
    b[left_nodes] = 1.0 * PENALTY
    b[right_nodes] = 0.0 * PENALTY # Redundant but clear

    print("  > Solving Linear System (This is the heavy part)...")
    V_flat = spsolve(A_final, b)
    V = V_flat.reshape((ny, nx))

    # Calculate Current
    # Current leaving col -2 entering col -1
    # I = (V[-2] - V[-1]) * G_horizontal

    v_left = V[:, -2]
    v_right = V[:, -1]

    # Re-calculate specific conductance at the drain boundary
    r_drain = (rho_map[:, -2] + rho_map[:, -1]) / 2.0
    diff_drain = (grid[:, -2] != grid[:, -1])
    r_drain[diff_drain] += r_interface
    g_drain = 1.0 / r_drain

    currents = (v_left - v_right) * g_drain
    total_current = np.sum(currents)

    rho_eff = (1.0 / total_current) * (ny / nx) # Geometric factor 1.0 for square
    return rho_eff, V

# --- 4. MAIN EXECUTION ---
print(f"--- SIMULATION STARTED: LARGE SCALE (1000x1000) ---")
print(f"Volume Fraction Sn: {VOL_FRAC_SN*100:.1f}%")

t0 = time.time()

# A. Initialize
grid = np.random.choice([0, 1], size=(N, N), p=[VOL_FRAC_SN, 1-VOL_FRAC_SN]).astype(np.int32)

# B. Grain Growth
print("Step 1: Growing Microstructure...")
final_grid = run_monte_carlo_fast(grid, MCS_STEPS, TEMPERATURE, N)
print(f"  > Growth Done ({time.time()-t0:.1f}s)")

# C. Physics Solve
print("Step 2: Solving Physics...")
rho, V = solve_resistivity_large_scale(final_grid, RHO_SN, RHO_BI, R_INT_VAL)

print("\n" + "="*50)
print(f"FINAL RESULT (N={N}x{N})")
print(f"Resistivity: {rho:.3f} uOhm-cm")
print(f"Total Time:  {(time.time()-t0)/60:.1f} Minutes")
print("="*50)

# D. Plotting
fig, ax = plt.subplots(1, 2, figsize=(18, 9))

# High-Res Microstructure
ax[0].imshow(final_grid, cmap='gray_r', interpolation='nearest')
ax[0].set_title(f"Microstructure (1 Million Atoms)\nBlack=Sn (Conductor), White=Bi")
ax[0].axis('off')

# Voltage Map
im = ax[1].imshow(V, cmap='magma', interpolation='bilinear')
ax[1].set_title(f"Voltage Potential Field\nGradient shows percolation path")
ax[1].axis('off')
plt.colorbar(im, ax=ax[1], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

# 7: Dual-Field Solid Solution Model.

In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as splinalg
import matplotlib.pyplot as plt
import time
from scipy.ndimage import gaussian_filter

# ==========================================
# 1. PHYSICS & GRID PARAMETERS
# ==========================================
GRID_SIZE = 500  # 500x500 grid for rapid and accurate testing
VOL_FRAC_SN = 0.49

# Phase Resistivities (micro-Ohm-cm)
RHO_SN = 18.0
RHO_BI_BASE = 129.0
INTERFACE_PENALTY = 45.0

# Solid Solution Parameters (The "Dirty Bismuth" Effect)
MAX_SOLUBILITY_SN_IN_BI = 0.02  # Maximum 2% Sn trapped in Bi grains
SCATTERING_PENALTY_K = 10000.0   # 1% Sn adds 100 micro-Ohm-cm to local resistivity

PHASE_BI = 0
PHASE_SN = 1

print("Initializing Dual-Field Microstructure...")
start_init = time.time()

# ==========================================
# 2. DUAL-FIELD MATRIX INITIALIZATION
# ==========================================
np.random.seed(42)
random_noise = np.random.rand(GRID_SIZE, GRID_SIZE)
smoothed = gaussian_filter(random_noise, sigma=3)
phase_matrix = np.where(smoothed > np.percentile(smoothed, (1-VOL_FRAC_SN)*100), PHASE_SN, PHASE_BI)

# The Concentration Field (Solute Trapping)
c_sn = np.zeros((GRID_SIZE, GRID_SIZE), dtype=float)
bi_mask = (phase_matrix == PHASE_BI)

# Normal distribution of trapped Sn, clipped between 0 and Max Solubility
trapped_sn = np.random.normal(loc=MAX_SOLUBILITY_SN_IN_BI/2, scale=0.005, size=np.sum(bi_mask))
trapped_sn = np.clip(trapped_sn, 0, MAX_SOLUBILITY_SN_IN_BI)
c_sn[bi_mask] = trapped_sn

# ==========================================
# 3. DYNAMIC RESISTIVITY MAPPING
# ==========================================
rho_map = np.zeros((GRID_SIZE, GRID_SIZE), dtype=float)
rho_map[phase_matrix == PHASE_SN] = RHO_SN
# Bismuth suffers from Solid Solution Hardening
rho_map[bi_mask] = RHO_BI_BASE + (SCATTERING_PENALTY_K * c_sn[bi_mask])

print(f"Average Bi phase resistivity increased from {RHO_BI_BASE} to {np.mean(rho_map[bi_mask]):.1f}")
print(f"Initialization finished in {time.time() - start_init:.2f} seconds.")

# ==========================================
# 4. FINITE DIFFERENCE SOLVER (VECTORIZED + FAST BOUNDARIES)
# ==========================================
print("\nBuilding Conductance Matrix (Vectorized)...")
start_build = time.time()

N = GRID_SIZE * GRID_SIZE

# --- Calculate all horizontal edges (Right) instantly ---
rho1_h, rho2_h = rho_map[:, :-1], rho_map[:, 1:]
p1_h, p2_h = phase_matrix[:, :-1], phase_matrix[:, 1:]
penalty_h = np.where(p1_h != p2_h, INTERFACE_PENALTY, 0.0)
g_h = 1.0 / ((rho1_h + rho2_h) / 2.0 + penalty_h)

# --- Calculate all vertical edges (Bottom) instantly ---
rho1_v, rho2_v = rho_map[:-1, :], rho_map[1:, :]
p1_v, p2_v = phase_matrix[:-1, :], phase_matrix[1:, :]
penalty_v = np.where(p1_v != p2_v, INTERFACE_PENALTY, 0.0)
g_v = 1.0 / ((rho1_v + rho2_v) / 2.0 + penalty_v)

# --- Build Diagonals ---
main_diag_2D = np.zeros((GRID_SIZE, GRID_SIZE))
main_diag_2D[:, :-1] += g_h  # Connect to right
main_diag_2D[:, 1:] += g_h   # Connect to left
main_diag_2D[:-1, :] += g_v  # Connect to bottom
main_diag_2D[1:, :] += g_v   # Connect to top
main_diag = main_diag_2D.flatten()

g_h_padded = np.zeros((GRID_SIZE, GRID_SIZE))
g_h_padded[:, :-1] = g_h
right_diag = -g_h_padded.flatten()[:-1]
bottom_diag = -g_v.flatten()

# Construct Sparse Matrix
offsets = [0, 1, GRID_SIZE]
A = sp.diags([main_diag, right_diag, bottom_diag], offsets, shape=(N, N), format='lil')
A = A + A.T - sp.diags(A.diagonal())

# EXTREMELY IMPORTANT FOR COLAB: Force matrix back to LIL format before altering rows!
# This prevents the slow "Changing sparsity structure" warning.
A = A.tolil()

# Apply Dirichlet Boundary Conditions (Left = 1V, Right = 0V)
b = np.zeros(N)
for y in range(GRID_SIZE):
    left_idx = y * GRID_SIZE
    right_idx = y * GRID_SIZE + (GRID_SIZE - 1)

    A[left_idx, :] = 0
    A[left_idx, left_idx] = 1.0
    b[left_idx] = 1.0

    A[right_idx, :] = 0
    A[right_idx, right_idx] = 1.0
    b[right_idx] = 0.0

A = A.tocsr()
print(f"Matrix built without warnings in {time.time() - start_build:.2f} seconds.")

# ==========================================
# 5. ITERATIVE SOLVER (COLAB SURVIVAL MODE)
# ==========================================
print("\nSolving PDE using Conjugate Gradient (Iterative Solver)...")
start_solve = time.time()

# CG is highly memory-efficient and mathematically optimal for this symmetric matrix
V, info = splinalg.cg(A, b, rtol=1e-8)

if info == 0:
    print(f"Solver converged successfully in {time.time() - start_solve:.1f} seconds.")
else:
    print(f"Solver stopped with code {info}. Results may be approximate.")

V_matrix = V.reshape((GRID_SIZE, GRID_SIZE))

# ==========================================
# 6. VECTORIZED CURRENT CALCULATION
# ==========================================
dV_right = V_matrix[:, -2] - V_matrix[:, -1]
g_right = g_h[:, -1]
total_current = np.sum(dV_right * g_right)

eff_resistivity = 1.0 / total_current
print(f"\n====================================================")
print(f">>> THEORETICAL MACROSCOPIC RESISTIVITY: {eff_resistivity:.2f} micro-Ohm-cm <<<")
print(f"====================================================\n")

# ==========================================
# 7. VISUALIZATION
# ==========================================
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

axs[0].imshow(phase_matrix, cmap='bone')
axs[0].set_title("Phase Morphology (Sn=White, Bi=Black)")

# This map will now show the gradient of trapped Tin inside the Bismuth
im1 = axs[1].imshow(rho_map, cmap='inferno')
axs[1].set_title("Local Resistivity Map (Solid Solution)")
fig.colorbar(im1, ax=axs[1], fraction=0.046, pad=0.04)

im2 = axs[2].imshow(V_matrix, cmap='magma')
axs[2].set_title(f"Voltage Field (Global $\\rho_{{eff}}$ = {eff_resistivity:.1f})")
fig.colorbar(im2, ax=axs[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

# Code 8- Numba-Accelerated KMC + Solute Segregation


In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as splinalg
import matplotlib.pyplot as plt
import time
from numba import njit
from scipy.ndimage import distance_transform_edt

# ==========================================
# 1. PHYSICS & GRID PARAMETERS
# ==========================================
GRID_SIZE = 500
VOL_FRAC_SN = 0.49

RHO_SN = 18.0
RHO_BI_BASE = 129.0
INTERFACE_PENALTY = 45.0

# Segregation Parameters (Snowplow Effect)
MAX_SOLUBILITY_SN_IN_BI = 0.02  # Max 2% Sn trapped at boundaries
SCATTERING_PENALTY_K = 10000.0
SEGREGATION_DEPTH = 5.0 # How many pixels deep the Tin gets trapped

PHASE_BI = 0
PHASE_SN = 1

# KMC Parameters
KMC_STEPS = 50
KT = 0.65  # Simulation temperature (controls grain roughness)
J = 1.0    # Interfacial energy cost

print("Initializing random melt...")
np.random.seed(42)
# Start with a completely random "liquid" mix
phase_matrix = np.where(np.random.rand(GRID_SIZE, GRID_SIZE) > (1-VOL_FRAC_SN), PHASE_SN, PHASE_BI).astype(np.int8)

# ==========================================
# 2. NUMBA KINETIC MONTE CARLO ENGINE (C-SPEED)
# ==========================================
@njit
def run_kmc_growth(matrix, steps, kT, J):
    rows, cols = matrix.shape
    N = rows * cols

    # Precompute energy probabilities for blazing fast lookups
    exp_table = np.zeros(9)
    for dE in range(9):
        exp_table[dE] = np.exp(-dE / kT)

    for step in range(steps):
        for _ in range(N):
            # Pick a random pixel
            x = np.random.randint(0, rows)
            y = np.random.randint(0, cols)

            # Pick a random neighbor (Von Neumann neighborhood)
            direction = np.random.randint(0, 4)
            nx, ny = x, y
            if direction == 0: nx = (x + 1) % rows
            elif direction == 1: nx = (x - 1) % rows
            elif direction == 2: ny = (y + 1) % cols
            elif direction == 3: ny = (y - 1) % cols

            current_state = matrix[x, y]
            new_state = matrix[nx, ny]

            if current_state == new_state:
                continue

            # Calculate local interfacial energy (Cost = J for every unlike neighbor)
            E_initial = 0
            E_final = 0

            # Check all 8 Moore neighbors
            for dx in (-1, 0, 1):
                for dy in (-1, 0, 1):
                    if dx == 0 and dy == 0: continue
                    nnx = (x + dx) % rows
                    nny = (y + dy) % cols
                    neighbor_state = matrix[nnx, nny]

                    if neighbor_state != current_state: E_initial += J
                    if neighbor_state != new_state: E_final += J

            dE = E_final - E_initial

            # Metropolis Criterion
            if dE <= 0:
                matrix[x, y] = new_state
            else:
                if np.random.rand() < exp_table[int(dE)]:
                    matrix[x, y] = new_state

    return matrix

print(f"Growing Grains via KMC ({KMC_STEPS} steps = {KMC_STEPS * GRID_SIZE * GRID_SIZE} flips)...")
start_kmc = time.time()
# The first time this runs, Numba takes ~2 seconds to compile the C-code
phase_matrix = run_kmc_growth(phase_matrix, KMC_STEPS, KT, J)
print(f"Grain growth completed in {time.time() - start_kmc:.2f} seconds.")

# ==========================================
# 3. SOLUTE SEGREGATION (THE "SNOWPLOW" EFFECT)
# ==========================================
print("Calculating Tin Segregation in Bismuth...")
# Find distance from every Bi pixel to the nearest Sn pixel
bi_mask = (phase_matrix == PHASE_BI)
sn_mask = (phase_matrix == PHASE_SN)

# distance_transform_edt gives the distance to the nearest 0.
# We want distance from Bi to Sn, so we pass the Bi mask (Sn is 0)
distance_to_sn = distance_transform_edt(bi_mask)

c_sn = np.zeros((GRID_SIZE, GRID_SIZE), dtype=float)

# Tin concentration decays exponentially the deeper you go into the Bi grain
# Boundary = High Tin (Max 2%). Core = Pure Bi (0%).
c_sn[bi_mask] = MAX_SOLUBILITY_SN_IN_BI * np.exp(-distance_to_sn[bi_mask] / SEGREGATION_DEPTH)

# Add a tiny bit of random thermal noise to the concentration
c_sn[bi_mask] += np.random.normal(0, 0.002, size=np.sum(bi_mask))
c_sn = np.clip(c_sn, 0, MAX_SOLUBILITY_SN_IN_BI)

# ==========================================
# 4. DYNAMIC RESISTIVITY MAPPING
# ==========================================
rho_map = np.zeros((GRID_SIZE, GRID_SIZE), dtype=float)
rho_map[phase_matrix == PHASE_SN] = RHO_SN
rho_map[bi_mask] = RHO_BI_BASE + (SCATTERING_PENALTY_K * c_sn[bi_mask])

print(f"Average Bi phase resistivity increased from {RHO_BI_BASE} to {np.mean(rho_map[bi_mask]):.1f}")

# ==========================================
# 5. FINITE DIFFERENCE SOLVER (VECTORIZED)
# ==========================================
print("Building Conductance Matrix...")
N = GRID_SIZE * GRID_SIZE

rho1_h, rho2_h = rho_map[:, :-1], rho_map[:, 1:]
p1_h, p2_h = phase_matrix[:, :-1], phase_matrix[:, 1:]
penalty_h = np.where(p1_h != p2_h, INTERFACE_PENALTY, 0.0)
g_h = 1.0 / ((rho1_h + rho2_h) / 2.0 + penalty_h)

rho1_v, rho2_v = rho_map[:-1, :], rho_map[1:, :]
p1_v, p2_v = phase_matrix[:-1, :], phase_matrix[1:, :]
penalty_v = np.where(p1_v != p2_v, INTERFACE_PENALTY, 0.0)
g_v = 1.0 / ((rho1_v + rho2_v) / 2.0 + penalty_v)

main_diag_2D = np.zeros((GRID_SIZE, GRID_SIZE))
main_diag_2D[:, :-1] += g_h
main_diag_2D[:, 1:] += g_h
main_diag_2D[:-1, :] += g_v
main_diag_2D[1:, :] += g_v
main_diag = main_diag_2D.flatten()

g_h_padded = np.zeros((GRID_SIZE, GRID_SIZE))
g_h_padded[:, :-1] = g_h
right_diag = -g_h_padded.flatten()[:-1]
bottom_diag = -g_v.flatten()

offsets = [0, 1, GRID_SIZE]
A = sp.diags([main_diag, right_diag, bottom_diag], offsets, shape=(N, N), format='lil')
A = A + A.T - sp.diags(A.diagonal())
A = A.tolil()

b = np.zeros(N)
for y in range(GRID_SIZE):
    left_idx = y * GRID_SIZE
    right_idx = y * GRID_SIZE + (GRID_SIZE - 1)
    A[left_idx, :] = 0
    A[left_idx, left_idx] = 1.0
    b[left_idx] = 1.0
    A[right_idx, :] = 0
    A[right_idx, right_idx] = 1.0
    b[right_idx] = 0.0

A = A.tocsr()

# ==========================================
# 6. SOLVER & CURRENT CALCULATION
# ==========================================
print("Solving PDE using Conjugate Gradient...")
V, info = splinalg.cg(A, b, rtol=1e-8)
V_matrix = V.reshape((GRID_SIZE, GRID_SIZE))

dV_right = V_matrix[:, -2] - V_matrix[:, -1]
g_right = g_h[:, -1]
total_current = np.sum(dV_right * g_right)
eff_resistivity = 1.0 / total_current

print(f"\n====================================================")
print(f">>> THEORETICAL MACROSCOPIC RESISTIVITY: {eff_resistivity:.2f} micro-Ohm-cm <<<")
print(f"====================================================\n")

# ==========================================
# 7. VISUALIZATION
# ==========================================
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

axs[0].imshow(phase_matrix, cmap='bone')
axs[0].set_title(f"KMC Grown Phase (MCS={KMC_STEPS})")

im1 = axs[1].imshow(rho_map, cmap='inferno')
axs[1].set_title("Resistivity Map (Boundary Segregation)")
fig.colorbar(im1, ax=axs[1], fraction=0.046, pad=0.04)

im2 = axs[2].imshow(V_matrix, cmap='magma')
axs[2].set_title(f"Voltage Field (Global $\\rho_{{eff}}$ = {eff_resistivity:.1f})")
fig.colorbar(im2, ax=axs[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

# Code 9- (Scaled Up)

In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as splinalg
import matplotlib.pyplot as plt
import time
from numba import njit
from scipy.ndimage import distance_transform_edt

# ==========================================
# 1. PHYSICS & HIGH-RES GRID PARAMETERS
# ==========================================
GRID_SIZE = 1000  # THE MEGA GRID (1 Million Pixels)
VOL_FRAC_SN = 0.49

RHO_SN = 18.0
RHO_BI_BASE = 129.0
INTERFACE_PENALTY = 45.0

# Segregation Parameters (Scaled for 1000x1000)
MAX_SOLUBILITY_SN_IN_BI = 0.02
SCATTERING_PENALTY_K = 10000.0
SEGREGATION_DEPTH = 10.0 # Doubled to match the new high-res scale

PHASE_BI = 0
PHASE_SN = 1

KMC_STEPS = 50
KT = 0.65
J = 1.0

print(f"Initializing {GRID_SIZE}x{GRID_SIZE} random melt...")
np.random.seed(42)
phase_matrix = np.where(np.random.rand(GRID_SIZE, GRID_SIZE) > (1-VOL_FRAC_SN), PHASE_SN, PHASE_BI).astype(np.int8)

# ==========================================
# 2. NUMBA KMC ENGINE (50 Million Flips)
# ==========================================
@njit
def run_kmc_growth(matrix, steps, kT, J):
    rows, cols = matrix.shape
    N = rows * cols

    exp_table = np.zeros(9)
    for dE in range(9):
        exp_table[dE] = np.exp(-dE / kT)

    for step in range(steps):
        for _ in range(N):
            x = np.random.randint(0, rows)
            y = np.random.randint(0, cols)

            direction = np.random.randint(0, 4)
            nx, ny = x, y
            if direction == 0: nx = (x + 1) % rows
            elif direction == 1: nx = (x - 1) % rows
            elif direction == 2: ny = (y + 1) % cols
            elif direction == 3: ny = (y - 1) % cols

            current_state = matrix[x, y]
            new_state = matrix[nx, ny]

            if current_state == new_state:
                continue

            E_initial = 0
            E_final = 0

            for dx in (-1, 0, 1):
                for dy in (-1, 0, 1):
                    if dx == 0 and dy == 0: continue
                    nnx = (x + dx) % rows
                    nny = (y + dy) % cols
                    neighbor_state = matrix[nnx, nny]

                    if neighbor_state != current_state: E_initial += J
                    if neighbor_state != new_state: E_final += J

            dE = E_final - E_initial

            if dE <= 0:
                matrix[x, y] = new_state
            else:
                if np.random.rand() < exp_table[int(dE)]:
                    matrix[x, y] = new_state

    return matrix

print(f"Growing Grains via KMC ({KMC_STEPS} steps = {KMC_STEPS * GRID_SIZE * GRID_SIZE} flips)...")
start_kmc = time.time()
phase_matrix = run_kmc_growth(phase_matrix, KMC_STEPS, KT, J)
print(f"Grain growth completed in {time.time() - start_kmc:.2f} seconds.")

# ==========================================
# 3. HIGH-RES SOLUTE SEGREGATION
# ==========================================
print("Calculating Tin Segregation in Bismuth...")
bi_mask = (phase_matrix == PHASE_BI)
sn_mask = (phase_matrix == PHASE_SN)

distance_to_sn = distance_transform_edt(bi_mask)
c_sn = np.zeros((GRID_SIZE, GRID_SIZE), dtype=float)

c_sn[bi_mask] = MAX_SOLUBILITY_SN_IN_BI * np.exp(-distance_to_sn[bi_mask] / SEGREGATION_DEPTH)
c_sn[bi_mask] += np.random.normal(0, 0.002, size=np.sum(bi_mask))
c_sn = np.clip(c_sn, 0, MAX_SOLUBILITY_SN_IN_BI)

# ==========================================
# 4. DYNAMIC RESISTIVITY MAPPING
# ==========================================
rho_map = np.zeros((GRID_SIZE, GRID_SIZE), dtype=float)
rho_map[phase_matrix == PHASE_SN] = RHO_SN
rho_map[bi_mask] = RHO_BI_BASE + (SCATTERING_PENALTY_K * c_sn[bi_mask])

print(f"Average Bi phase resistivity increased from {RHO_BI_BASE} to {np.mean(rho_map[bi_mask]):.1f}")

# ==========================================
# 5. FINITE DIFFERENCE SOLVER (1 Million Nodes)
# ==========================================
print("Building 1-Million Node Conductance Matrix...")
start_build = time.time()
N = GRID_SIZE * GRID_SIZE

rho1_h, rho2_h = rho_map[:, :-1], rho_map[:, 1:]
p1_h, p2_h = phase_matrix[:, :-1], phase_matrix[:, 1:]
penalty_h = np.where(p1_h != p2_h, INTERFACE_PENALTY, 0.0)
g_h = 1.0 / ((rho1_h + rho2_h) / 2.0 + penalty_h)

rho1_v, rho2_v = rho_map[:-1, :], rho_map[1:, :]
p1_v, p2_v = phase_matrix[:-1, :], phase_matrix[1:, :]
penalty_v = np.where(p1_v != p2_v, INTERFACE_PENALTY, 0.0)
g_v = 1.0 / ((rho1_v + rho2_v) / 2.0 + penalty_v)

main_diag_2D = np.zeros((GRID_SIZE, GRID_SIZE))
main_diag_2D[:, :-1] += g_h
main_diag_2D[:, 1:] += g_h
main_diag_2D[:-1, :] += g_v
main_diag_2D[1:, :] += g_v
main_diag = main_diag_2D.flatten()

g_h_padded = np.zeros((GRID_SIZE, GRID_SIZE))
g_h_padded[:, :-1] = g_h
right_diag = -g_h_padded.flatten()[:-1]
bottom_diag = -g_v.flatten()

offsets = [0, 1, GRID_SIZE]
A = sp.diags([main_diag, right_diag, bottom_diag], offsets, shape=(N, N), format='lil')
A = A + A.T - sp.diags(A.diagonal())
A = A.tolil()

b = np.zeros(N)
for y in range(GRID_SIZE):
    left_idx = y * GRID_SIZE
    right_idx = y * GRID_SIZE + (GRID_SIZE - 1)
    A[left_idx, :] = 0
    A[left_idx, left_idx] = 1.0
    b[left_idx] = 1.0
    A[right_idx, :] = 0
    A[right_idx, right_idx] = 1.0
    b[right_idx] = 0.0

A = A.tocsr()
print(f"Matrix built in {time.time() - start_build:.2f} seconds.")

# ==========================================
# 6. ITERATIVE SOLVER (Colab Safe)
# ==========================================
print("Solving PDE using Conjugate Gradient...")
start_solve = time.time()
V, info = splinalg.cg(A, b, rtol=1e-8)
print(f"Solver finished in {time.time() - start_solve:.2f} seconds.")

V_matrix = V.reshape((GRID_SIZE, GRID_SIZE))

dV_right = V_matrix[:, -2] - V_matrix[:, -1]
g_right = g_h[:, -1]
total_current = np.sum(dV_right * g_right)
eff_resistivity = 1.0 / total_current

print(f"\n====================================================")
print(f">>> 1000x1000 MACROSCOPIC RESISTIVITY: {eff_resistivity:.2f} micro-Ohm-cm <<<")
print(f"====================================================\n")

# ==========================================
# 7. HIGH-RES VISUALIZATION
# ==========================================
# Increased figsize for presentation-quality output
fig, axs = plt.subplots(1, 3, figsize=(24, 8))

axs[0].imshow(phase_matrix, cmap='bone')
axs[0].set_title(f"High-Res KMC Grown Phase (1M Pixels)", fontsize=14)
axs[0].axis('off') # Turn off axes for cleaner presentation images

im1 = axs[1].imshow(rho_map, cmap='inferno')
axs[1].set_title("Resistivity Map (Boundary Segregation)", fontsize=14)
axs[1].axis('off')
fig.colorbar(im1, ax=axs[1], fraction=0.046, pad=0.04)

im2 = axs[2].imshow(V_matrix, cmap='magma')
axs[2].set_title(f"Voltage Field (Global $\\rho_{{eff}}$ = {eff_resistivity:.1f})", fontsize=14)
axs[2].axis('off')
fig.colorbar(im2, ax=axs[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

# Code *10* Dynamic $\rho(C)$ and Mutual Solubility

In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as splinalg
import matplotlib.pyplot as plt
import time
from numba import njit
from scipy.ndimage import distance_transform_edt

# ==========================================
# 1. PHYSICS & HIGH-RES GRID PARAMETERS
# ==========================================
GRID_SIZE = 1000
VOL_FRAC_SN = 0.49
INTERFACE_PENALTY = 45.0

# --- NEW: Absolute Pure Elemental Resistivities ---
RHO_SN_PURE = 15.0   # Theoretical absolute pure Tin
RHO_BI_PURE = 115.0  # Theoretical absolute pure Bismuth

# --- NEW: Baseline Solid Solubilities at Room Temp ---
BASE_SOL_BI_IN_SN = 0.03   # ~3% Bismuth exists everywhere in Tin
BASE_SOL_SN_IN_BI = 0.001  # ~0.1% Tin exists everywhere in Bismuth

# --- NEW: Boundary Segregation Limits (High-Temp cooling) ---
MAX_SEGREGATION_BI_IN_SN = 0.21 # Bi precipitates up to 21% at Sn boundaries
MAX_SEGREGATION_SN_IN_BI = 0.02 # Sn segregates up to 2% at Bi boundaries
SEGREGATION_DEPTH = 10.0

# --- NEW: Nordheim Scattering Coefficients (k) ---
K_BI_IN_SN = 100.0    # Linear penalty for Bi atoms scattering in Sn lattice
K_SN_IN_BI = 10000.0  # Massive penalty for Sn atoms in Bi lattice (Dirty Bismuth)

PHASE_BI = 0
PHASE_SN = 1

KMC_STEPS = 50
KT = 0.65
J = 1.0

print(f"Initializing {GRID_SIZE}x{GRID_SIZE} random melt...")
np.random.seed(42)
phase_matrix = np.where(np.random.rand(GRID_SIZE, GRID_SIZE) > (1-VOL_FRAC_SN), PHASE_SN, PHASE_BI).astype(np.int8)

# ==========================================
# 2. NUMBA KMC ENGINE (Grain Growth)
# ==========================================
@njit
def run_kmc_growth(matrix, steps, kT, J):
    rows, cols = matrix.shape
    N = rows * cols

    exp_table = np.zeros(9)
    for dE in range(9):
        exp_table[dE] = np.exp(-dE / kT)

    for step in range(steps):
        for _ in range(N):
            x = np.random.randint(0, rows)
            y = np.random.randint(0, cols)

            direction = np.random.randint(0, 4)
            nx, ny = x, y
            if direction == 0: nx = (x + 1) % rows
            elif direction == 1: nx = (x - 1) % rows
            elif direction == 2: ny = (y + 1) % cols
            elif direction == 3: ny = (y - 1) % cols

            current_state = matrix[x, y]
            new_state = matrix[nx, ny]

            if current_state == new_state: continue

            E_initial, E_final = 0, 0

            for dx in (-1, 0, 1):
                for dy in (-1, 0, 1):
                    if dx == 0 and dy == 0: continue
                    nnx = (x + dx) % rows
                    nny = (y + dy) % cols
                    neighbor_state = matrix[nnx, nny]

                    if neighbor_state != current_state: E_initial += J
                    if neighbor_state != new_state: E_final += J

            dE = E_final - E_initial

            if dE <= 0:
                matrix[x, y] = new_state
            else:
                if np.random.rand() < exp_table[int(dE)]:
                    matrix[x, y] = new_state

    return matrix

print(f"Growing Grains via KMC ({KMC_STEPS} steps)...")
start_kmc = time.time()
phase_matrix = run_kmc_growth(phase_matrix, KMC_STEPS, KT, J)
print(f"Grain growth completed in {time.time() - start_kmc:.2f} seconds.")

# ==========================================
# 3. DUAL CONCENTRATION FIELDS (Mutual Solubility)
# ==========================================
print("Calculating Continuous Solid Solution Concentration Fields...")
bi_mask = (phase_matrix == PHASE_BI)
sn_mask = (phase_matrix == PHASE_SN)

# Calculate distances to opposing phases
distance_to_sn = distance_transform_edt(bi_mask)
distance_to_bi = distance_transform_edt(sn_mask)

# Initialize concentration matrices
c_sn_impurity = np.zeros((GRID_SIZE, GRID_SIZE), dtype=float)
c_bi_impurity = np.zeros((GRID_SIZE, GRID_SIZE), dtype=float)

# 3a. Sn impurities inside Bismuth (Baseline + Boundary Segregation)
c_sn_impurity[bi_mask] = BASE_SOL_SN_IN_BI + (MAX_SEGREGATION_SN_IN_BI - BASE_SOL_SN_IN_BI) * np.exp(-distance_to_sn[bi_mask] / SEGREGATION_DEPTH)
c_sn_impurity[bi_mask] += np.random.normal(0, 0.0005, size=np.sum(bi_mask)) # Add slight physical noise
c_sn_impurity = np.clip(c_sn_impurity, 0, MAX_SEGREGATION_SN_IN_BI)

# 3b. Bi impurities inside Tin (Baseline + Boundary Segregation)
c_bi_impurity[sn_mask] = BASE_SOL_BI_IN_SN + (MAX_SEGREGATION_BI_IN_SN - BASE_SOL_BI_IN_SN) * np.exp(-distance_to_bi[sn_mask] / SEGREGATION_DEPTH)
c_bi_impurity[sn_mask] += np.random.normal(0, 0.002, size=np.sum(sn_mask))
c_bi_impurity = np.clip(c_bi_impurity, 0, MAX_SEGREGATION_BI_IN_SN)

# ==========================================
# 4. VECTORIZED DYNAMIC RESISTIVITY p(C)
# ==========================================
print("Applying Nordheim's Rule for Dynamic Resistivity...")
rho_map = np.zeros((GRID_SIZE, GRID_SIZE), dtype=float)

# Evaluate p(C) = p_pure + k * C directly using vectorized math
rho_map[sn_mask] = RHO_SN_PURE + (K_BI_IN_SN * c_bi_impurity[sn_mask])
rho_map[bi_mask] = RHO_BI_PURE + (K_SN_IN_BI * c_sn_impurity[bi_mask])

print(f"Average Sn phase resistivity: {np.mean(rho_map[sn_mask]):.1f} (Theoretical Pure: {RHO_SN_PURE})")
print(f"Average Bi phase resistivity: {np.mean(rho_map[bi_mask]):.1f} (Theoretical Pure: {RHO_BI_PURE})")

# ==========================================
# 5. FINITE DIFFERENCE SOLVER
# ==========================================
print("Building 1-Million Node Conductance Matrix...")
start_build = time.time()
N = GRID_SIZE * GRID_SIZE

rho1_h, rho2_h = rho_map[:, :-1], rho_map[:, 1:]
p1_h, p2_h = phase_matrix[:, :-1], phase_matrix[:, 1:]
penalty_h = np.where(p1_h != p2_h, INTERFACE_PENALTY, 0.0)
g_h = 1.0 / ((rho1_h + rho2_h) / 2.0 + penalty_h)

rho1_v, rho2_v = rho_map[:-1, :], rho_map[1:, :]
p1_v, p2_v = phase_matrix[:-1, :], phase_matrix[1:, :]
penalty_v = np.where(p1_v != p2_v, INTERFACE_PENALTY, 0.0)
g_v = 1.0 / ((rho1_v + rho2_v) / 2.0 + penalty_v)

main_diag_2D = np.zeros((GRID_SIZE, GRID_SIZE))
main_diag_2D[:, :-1] += g_h
main_diag_2D[:, 1:] += g_h
main_diag_2D[:-1, :] += g_v
main_diag_2D[1:, :] += g_v
main_diag = main_diag_2D.flatten()

g_h_padded = np.zeros((GRID_SIZE, GRID_SIZE))
g_h_padded[:, :-1] = g_h
right_diag = -g_h_padded.flatten()[:-1]
bottom_diag = -g_v.flatten()

offsets = [0, 1, GRID_SIZE]
A = sp.diags([main_diag, right_diag, bottom_diag], offsets, shape=(N, N), format='lil')
A = A + A.T - sp.diags(A.diagonal())
A = A.tolil()

b = np.zeros(N)
for y in range(GRID_SIZE):
    left_idx = y * GRID_SIZE
    right_idx = y * GRID_SIZE + (GRID_SIZE - 1)
    A[left_idx, :] = 0
    A[left_idx, left_idx] = 1.0
    b[left_idx] = 1.0
    A[right_idx, :] = 0
    A[right_idx, right_idx] = 1.0
    b[right_idx] = 0.0

A = A.tocsr()
print(f"Matrix built in {time.time() - start_build:.2f} seconds.")

# ==========================================
# 6. ITERATIVE SOLVER
# ==========================================
print("Solving PDE using Conjugate Gradient...")
start_solve = time.time()
V, info = splinalg.cg(A, b, rtol=1e-8)
print(f"Solver finished in {time.time() - start_solve:.2f} seconds.")

V_matrix = V.reshape((GRID_SIZE, GRID_SIZE))

dV_right = V_matrix[:, -2] - V_matrix[:, -1]
g_right = g_h[:, -1]
total_current = np.sum(dV_right * g_right)
eff_resistivity = 1.0 / total_current

print(f"\n====================================================")
print(f">>> CONTINUUM MODEL MACROSCOPIC RESISTIVITY: {eff_resistivity:.2f} micro-Ohm-cm <<<")
print(f"====================================================\n")

# ==========================================
# 7. HIGH-RES VISUALIZATION
# ==========================================
fig, axs = plt.subplots(1, 3, figsize=(24, 8))

axs[0].imshow(phase_matrix, cmap='bone')
axs[0].set_title("High-Res Phase Matrix", fontsize=14)
axs[0].axis('off')

im1 = axs[1].imshow(rho_map, cmap='inferno')
axs[1].set_title(r"Dynamic $\rho(C)$ Resistivity Map", fontsize=14)
axs[1].axis('off')
fig.colorbar(im1, ax=axs[1], fraction=0.046, pad=0.04)

im2 = axs[2].imshow(V_matrix, cmap='magma')
axs[2].set_title(f"Voltage Field (Global $\\rho_{{eff}}$ = {eff_resistivity:.1f})", fontsize=14)
axs[2].axis('off')
fig.colorbar(im2, ax=axs[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

# Code 10.1: The Tuned Continuum Model

In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as splinalg
import matplotlib.pyplot as plt
import time
from numba import njit
from scipy.ndimage import distance_transform_edt

# ==========================================
# 1. PHYSICS & HIGH-RES GRID PARAMETERS (10.2 DELTA-C MODEL)
# ==========================================
GRID_SIZE = 1000
VOL_FRAC_SN = 0.49
INTERFACE_PENALTY = 45.0

# EMPIRICAL BASELINE RESISTIVITIES (Anchored to Room Temp Reality)
RHO_SN_BASE = 18.0
RHO_BI_BASE = 129.0

# Baseline Solid Solubilities at Room Temp
BASE_SOL_BI_IN_SN = 0.03   # 3% Bi already baked into the 18.0 baseline [cite: 5]
BASE_SOL_SN_IN_BI = 0.001  # 0.1% Sn already baked into the 129.0 baseline

# Boundary Segregation Limits
MAX_SEGREGATION_BI_IN_SN = 0.21 # Solvus limit at high temp [cite: 5]
MAX_SEGREGATION_SN_IN_BI = 0.02 # Kinetic trapping limit
SEGREGATION_DEPTH = 8.0

# Nordheim Scattering Coefficients (k)
K_BI_IN_SN = 20.0
K_SN_IN_BI = 10000.0

PHASE_BI = 0
PHASE_SN = 1

KMC_STEPS = 50
KT = 0.65
J = 1.0

print(f"Initializing {GRID_SIZE}x{GRID_SIZE} random melt...")
np.random.seed(42)
phase_matrix = np.where(np.random.rand(GRID_SIZE, GRID_SIZE) > (1-VOL_FRAC_SN), PHASE_SN, PHASE_BI).astype(np.int8)

# ==========================================
# 2. NUMBA KMC ENGINE (Quenching the Melt)
# ==========================================
@njit
def run_kmc_growth(matrix, steps, kT, J):
    rows, cols = matrix.shape
    N = rows * cols

    exp_table = np.zeros(9)
    for dE in range(9):
        exp_table[dE] = np.exp(-dE / kT)

    for step in range(steps):
        for _ in range(N):
            x = np.random.randint(0, rows)
            y = np.random.randint(0, cols)

            direction = np.random.randint(0, 4)
            nx, ny = x, y
            if direction == 0: nx = (x + 1) % rows
            elif direction == 1: nx = (x - 1) % rows
            elif direction == 2: ny = (y + 1) % cols
            elif direction == 3: ny = (y - 1) % cols

            current_state = matrix[x, y]
            new_state = matrix[nx, ny]

            if current_state == new_state: continue

            E_initial, E_final = 0, 0

            for dx in (-1, 0, 1):
                for dy in (-1, 0, 1):
                    if dx == 0 and dy == 0: continue
                    nnx = (x + dx) % rows
                    nny = (y + dy) % cols
                    neighbor_state = matrix[nnx, nny]

                    if neighbor_state != current_state: E_initial += J
                    if neighbor_state != new_state: E_final += J

            dE = E_final - E_initial

            if dE <= 0:
                matrix[x, y] = new_state
            else:
                if np.random.rand() < exp_table[int(dE)]:
                    matrix[x, y] = new_state

    return matrix

print(f"Growing Grains via KMC ({KMC_STEPS} steps)...")
start_kmc = time.time()
phase_matrix = run_kmc_growth(phase_matrix, KMC_STEPS, KT, J)
print(f"Grain growth completed in {time.time() - start_kmc:.2f} seconds.")

# ==========================================
# 3. DUAL CONCENTRATION FIELDS
# ==========================================
print("Calculating Continuous Solid Solution Concentration Fields...")
bi_mask = (phase_matrix == PHASE_BI)
sn_mask = (phase_matrix == PHASE_SN)

distance_to_sn = distance_transform_edt(bi_mask)
distance_to_bi = distance_transform_edt(sn_mask)

c_sn_impurity = np.zeros((GRID_SIZE, GRID_SIZE), dtype=float)
c_bi_impurity = np.zeros((GRID_SIZE, GRID_SIZE), dtype=float)

c_sn_impurity[bi_mask] = BASE_SOL_SN_IN_BI + (MAX_SEGREGATION_SN_IN_BI - BASE_SOL_SN_IN_BI) * np.exp(-distance_to_sn[bi_mask] / SEGREGATION_DEPTH)
c_sn_impurity[bi_mask] += np.random.normal(0, 0.0005, size=np.sum(bi_mask))
c_sn_impurity = np.clip(c_sn_impurity, 0, MAX_SEGREGATION_SN_IN_BI)

c_bi_impurity[sn_mask] = BASE_SOL_BI_IN_SN + (MAX_SEGREGATION_BI_IN_SN - BASE_SOL_BI_IN_SN) * np.exp(-distance_to_bi[sn_mask] / SEGREGATION_DEPTH)
c_bi_impurity[sn_mask] += np.random.normal(0, 0.002, size=np.sum(sn_mask))
c_bi_impurity = np.clip(c_bi_impurity, 0, MAX_SEGREGATION_BI_IN_SN)

# ==========================================
# 4. VECTORIZED DYNAMIC RESISTIVITY (THE DELTA-C UPGRADE)
# ==========================================
print("Applying Nordheim's Rule (Delta-C Offset)...")
rho_map = np.zeros((GRID_SIZE, GRID_SIZE), dtype=float)

# Calculate the *excess* concentration above the room-temp baseline
delta_c_bi_in_sn = np.maximum(0, c_bi_impurity[sn_mask] - BASE_SOL_BI_IN_SN)
delta_c_sn_in_bi = np.maximum(0, c_sn_impurity[bi_mask] - BASE_SOL_SN_IN_BI)

# Apply scattering penalty ONLY to the excess trapped impurities
rho_map[sn_mask] = RHO_SN_BASE + (K_BI_IN_SN * delta_c_bi_in_sn)
rho_map[bi_mask] = RHO_BI_BASE + (K_SN_IN_BI * delta_c_sn_in_bi)

print(f"Average Sn phase resistivity: {np.mean(rho_map[sn_mask]):.1f} (Anchored to: {RHO_SN_BASE})")
print(f"Average Bi phase resistivity: {np.mean(rho_map[bi_mask]):.1f} (Anchored to: {RHO_BI_BASE})")

# ==========================================
# 5. FINITE DIFFERENCE SOLVER
# ==========================================
print("Building 1-Million Node Conductance Matrix...")
start_build = time.time()
N = GRID_SIZE * GRID_SIZE

rho1_h, rho2_h = rho_map[:, :-1], rho_map[:, 1:]
p1_h, p2_h = phase_matrix[:, :-1], phase_matrix[:, 1:]
penalty_h = np.where(p1_h != p2_h, INTERFACE_PENALTY, 0.0)
g_h = 1.0 / ((rho1_h + rho2_h) / 2.0 + penalty_h)

rho1_v, rho2_v = rho_map[:-1, :], rho_map[1:, :]
p1_v, p2_v = phase_matrix[:-1, :], phase_matrix[1:, :]
penalty_v = np.where(p1_v != p2_v, INTERFACE_PENALTY, 0.0)
g_v = 1.0 / ((rho1_v + rho2_v) / 2.0 + penalty_v)

main_diag_2D = np.zeros((GRID_SIZE, GRID_SIZE))
main_diag_2D[:, :-1] += g_h
main_diag_2D[:, 1:] += g_h
main_diag_2D[:-1, :] += g_v
main_diag_2D[1:, :] += g_v
main_diag = main_diag_2D.flatten()

g_h_padded = np.zeros((GRID_SIZE, GRID_SIZE))
g_h_padded[:, :-1] = g_h
right_diag = -g_h_padded.flatten()[:-1]
bottom_diag = -g_v.flatten()

offsets = [0, 1, GRID_SIZE]
A = sp.diags([main_diag, right_diag, bottom_diag], offsets, shape=(N, N), format='lil')
A = A + A.T - sp.diags(A.diagonal())
A = A.tolil()

b = np.zeros(N)
for y in range(GRID_SIZE):
    left_idx = y * GRID_SIZE
    right_idx = y * GRID_SIZE + (GRID_SIZE - 1)
    A[left_idx, :] = 0
    A[left_idx, left_idx] = 1.0
    b[left_idx] = 1.0
    A[right_idx, :] = 0
    A[right_idx, right_idx] = 1.0
    b[right_idx] = 0.0

A = A.tocsr()
print(f"Matrix built in {time.time() - start_build:.2f} seconds.")

# ==========================================
# 6. ITERATIVE SOLVER
# ==========================================
print("Solving PDE using Conjugate Gradient...")
start_solve = time.time()
V, info = splinalg.cg(A, b, rtol=1e-8)
print(f"Solver finished in {time.time() - start_solve:.2f} seconds.")

V_matrix = V.reshape((GRID_SIZE, GRID_SIZE))

dV_right = V_matrix[:, -2] - V_matrix[:, -1]
g_right = g_h[:, -1]
total_current = np.sum(dV_right * g_right)
eff_resistivity = 1.0 / total_current

print(f"\n====================================================")
print(f">>> DELTA-C MODEL MACROSCOPIC RESISTIVITY: {eff_resistivity:.2f} micro-Ohm-cm <<<")
print(f"====================================================\n")

# ==========================================
# 7. HIGH-RES VISUALIZATION
# ==========================================
fig, axs = plt.subplots(1, 3, figsize=(24, 8))

axs[0].imshow(phase_matrix, cmap='bone')
axs[0].set_title("High-Res Phase Matrix", fontsize=14)
axs[0].axis('off')

im1 = axs[1].imshow(rho_map, cmap='inferno')
axs[1].set_title(r"Dynamic $\rho(\Delta C)$ Resistivity Map", fontsize=14)
axs[1].axis('off')
fig.colorbar(im1, ax=axs[1], fraction=0.046, pad=0.04)

im2 = axs[2].imshow(V_matrix, cmap='magma')
axs[2].set_title(f"Voltage Field (Global $\\rho_{{eff}}$ = {eff_resistivity:.1f})", fontsize=14)
axs[2].axis('off')
fig.colorbar(im2, ax=axs[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

# Code 10.3 Compositonal Gradient

In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as splinalg
import matplotlib.pyplot as plt
import time
from numba import njit
from scipy.ndimage import convolve

# ==========================================
# 1. PHYSICS & HIGH-RES GRID PARAMETERS
# ==========================================
GRID_SIZE = 1000
VOL_FRAC_SN = 0.49
INTERFACE_PENALTY = 45.0

# EMPIRICAL BASELINE RESISTIVITIES
RHO_SN_BASE = 18.0
RHO_BI_BASE = 129.0

# Baseline Solid Solubilities at Room Temp
BASE_SOL_BI_IN_SN = 0.03
BASE_SOL_SN_IN_BI = 0.001

# Boundary Segregation Limits (High Temp / Trapping)
MAX_SEGREGATION_BI_IN_SN = 0.21
MAX_SEGREGATION_SN_IN_BI = 0.02

# Nordheim Scattering Coefficients (k)
K_BI_IN_SN = 20.0
K_SN_IN_BI = 10000.0

PHASE_BI = 0
PHASE_SN = 1

KMC_STEPS = 50
KT = 0.65
J = 1.0

print(f"Initializing {GRID_SIZE}x{GRID_SIZE} random melt...")
np.random.seed(42)
phase_matrix = np.where(np.random.rand(GRID_SIZE, GRID_SIZE) > (1-VOL_FRAC_SN), PHASE_SN, PHASE_BI).astype(np.int8)

# ==========================================
# 2. NUMBA KMC ENGINE (Quenching the Melt)
# ==========================================
@njit
def run_kmc_growth(matrix, steps, kT, J):
    rows, cols = matrix.shape
    N = rows * cols
    exp_table = np.zeros(9)
    for dE in range(9):
        exp_table[dE] = np.exp(-dE / kT)

    for step in range(steps):
        for _ in range(N):
            x = np.random.randint(0, rows)
            y = np.random.randint(0, cols)

            direction = np.random.randint(0, 4)
            nx, ny = x, y
            if direction == 0: nx = (x + 1) % rows
            elif direction == 1: nx = (x - 1) % rows
            elif direction == 2: ny = (y + 1) % cols
            elif direction == 3: ny = (y - 1) % cols

            current_state = matrix[x, y]
            new_state = matrix[nx, ny]

            if current_state == new_state: continue

            E_initial, E_final = 0, 0
            for dx in (-1, 0, 1):
                for dy in (-1, 0, 1):
                    if dx == 0 and dy == 0: continue
                    nnx = (x + dx) % rows
                    nny = (y + dy) % cols
                    neighbor_state = matrix[nnx, nny]
                    if neighbor_state != current_state: E_initial += J
                    if neighbor_state != new_state: E_final += J

            dE = E_final - E_initial
            if dE <= 0:
                matrix[x, y] = new_state
            else:
                if np.random.rand() < exp_table[int(dE)]:
                    matrix[x, y] = new_state

    return matrix

print(f"Growing Grains via KMC ({KMC_STEPS} steps)...")
start_kmc = time.time()
phase_matrix = run_kmc_growth(phase_matrix, KMC_STEPS, KT, J)
print(f"Grain growth completed in {time.time() - start_kmc:.2f} seconds.")

# ==========================================
# 3. THERMODYNAMIC COMPOSITIONAL GRADIENTS (FICK'S 2ND LAW)
# ==========================================
print("Calculating Thermodynamic Composition Gradients via Fick's 2nd Law...")
start_diff = time.time()

bi_mask = (phase_matrix == PHASE_BI)
sn_mask = (phase_matrix == PHASE_SN)

# Initialize concentration fields at their room-temp baselines
c_sn_impurity = np.full((GRID_SIZE, GRID_SIZE), BASE_SOL_SN_IN_BI)
c_bi_impurity = np.full((GRID_SIZE, GRID_SIZE), BASE_SOL_BI_IN_SN)

# 2D Laplacian Kernel to calculate spatial gradients (∇²C)
laplacian_kernel = np.array([[0,  1,  0],
                             [1, -4,  1],
                             [0,  1,  0]])

# Detect interfacial grain boundaries
sn_edges = np.logical_and(sn_mask, convolve(sn_mask.astype(int), laplacian_kernel, mode='constant') < 0)
bi_edges = np.logical_and(bi_mask, convolve(bi_mask.astype(int), laplacian_kernel, mode='constant') < 0)

# Thermodynamic Diffusion Parameters
D_bi = 0.15   # Diffusion coefficient for Bi in Sn
D_sn = 0.10   # Diffusion coefficient for Sn in Bi
dt = 0.1      # Time step
diffusion_steps = 150  # Number of cooling iterations

# Run Fick's Second Law PDE
for _ in range(diffusion_steps):
    nabla2_c_bi = convolve(c_bi_impurity, laplacian_kernel, mode='nearest')
    nabla2_c_sn = convolve(c_sn_impurity, laplacian_kernel, mode='nearest')

    # Apply Euler step: dC/dt = D * ∇²C
    c_bi_impurity[sn_mask] += (D_bi * nabla2_c_bi * dt)[sn_mask]
    c_sn_impurity[bi_mask] += (D_sn * nabla2_c_sn * dt)[bi_mask]

    # Enforce Dirichlet Boundary Conditions (Continuous precipitation pinning)
    c_bi_impurity[sn_edges] = MAX_SEGREGATION_BI_IN_SN
    c_sn_impurity[bi_edges] = MAX_SEGREGATION_SN_IN_BI

# Add minor thermodynamic noise (thermal fluctuations)
c_sn_impurity[bi_mask] += np.random.normal(0, 0.0002, size=np.sum(bi_mask))
c_bi_impurity[sn_mask] += np.random.normal(0, 0.0005, size=np.sum(sn_mask))

print(f"Diffusion gradients resolved in {time.time() - start_diff:.2f} seconds.")

# ==========================================
# 4. VECTORIZED DYNAMIC RESISTIVITY (THE DELTA-C UPGRADE)
# ==========================================
print("Applying Nordheim's Rule (Delta-C Offset)...")
rho_map = np.zeros((GRID_SIZE, GRID_SIZE), dtype=float)

# Calculate exactly how much excess impurity exists above baseline
delta_c_bi_in_sn = np.maximum(0, c_bi_impurity[sn_mask] - BASE_SOL_BI_IN_SN)
delta_c_sn_in_bi = np.maximum(0, c_sn_impurity[bi_mask] - BASE_SOL_SN_IN_BI)

rho_map[sn_mask] = RHO_SN_BASE + (K_BI_IN_SN * delta_c_bi_in_sn)
rho_map[bi_mask] = RHO_BI_BASE + (K_SN_IN_BI * delta_c_sn_in_bi)

# ==========================================
# 5. FINITE DIFFERENCE SOLVER
# ==========================================
print("Building 1-Million Node Conductance Matrix...")
start_build = time.time()
N = GRID_SIZE * GRID_SIZE

rho1_h, rho2_h = rho_map[:, :-1], rho_map[:, 1:]
p1_h, p2_h = phase_matrix[:, :-1], phase_matrix[:, 1:]
penalty_h = np.where(p1_h != p2_h, INTERFACE_PENALTY, 0.0)
g_h = 1.0 / ((rho1_h + rho2_h) / 2.0 + penalty_h)

rho1_v, rho2_v = rho_map[:-1, :], rho_map[1:, :]
p1_v, p2_v = phase_matrix[:-1, :], phase_matrix[1:, :]
penalty_v = np.where(p1_v != p2_v, INTERFACE_PENALTY, 0.0)
g_v = 1.0 / ((rho1_v + rho2_v) / 2.0 + penalty_v)

main_diag_2D = np.zeros((GRID_SIZE, GRID_SIZE))
main_diag_2D[:, :-1] += g_h
main_diag_2D[:, 1:] += g_h
main_diag_2D[:-1, :] += g_v
main_diag_2D[1:, :] += g_v
main_diag = main_diag_2D.flatten()

g_h_padded = np.zeros((GRID_SIZE, GRID_SIZE))
g_h_padded[:, :-1] = g_h
right_diag = -g_h_padded.flatten()[:-1]
bottom_diag = -g_v.flatten()

offsets = [0, 1, GRID_SIZE]
A = sp.diags([main_diag, right_diag, bottom_diag], offsets, shape=(N, N), format='lil')
A = A + A.T - sp.diags(A.diagonal())
A = A.tolil()

b = np.zeros(N)
for y in range(GRID_SIZE):
    left_idx = y * GRID_SIZE
    right_idx = y * GRID_SIZE + (GRID_SIZE - 1)
    A[left_idx, :] = 0
    A[left_idx, left_idx] = 1.0
    b[left_idx] = 1.0
    A[right_idx, :] = 0
    A[right_idx, right_idx] = 1.0
    b[right_idx] = 0.0

A = A.tocsr()
print(f"Matrix built in {time.time() - start_build:.2f} seconds.")

# ==========================================
# 6. ITERATIVE SOLVER
# ==========================================
print("Solving PDE using Conjugate Gradient...")
start_solve = time.time()
V, info = splinalg.cg(A, b, rtol=1e-8)
print(f"Solver finished in {time.time() - start_solve:.2f} seconds.")

V_matrix = V.reshape((GRID_SIZE, GRID_SIZE))

dV_right = V_matrix[:, -2] - V_matrix[:, -1]
g_right = g_h[:, -1]
total_current = np.sum(dV_right * g_right)
eff_resistivity = 1.0 / total_current

print(f"\n====================================================")
print(f">>> FICK'S DIFFUSION MACROSCOPIC RESISTIVITY: {eff_resistivity:.2f} micro-Ohm-cm <<<")
print(f"====================================================\n")

# ==========================================
# 7. HIGH-RES VISUALIZATION (THE DELIVERABLE)
# ==========================================
# Combine the two impurity fields into one master composition map for the plot
composition_map = np.zeros((GRID_SIZE, GRID_SIZE), dtype=float)
composition_map[sn_mask] = c_bi_impurity[sn_mask]  # Bi diffusing into Sn
composition_map[bi_mask] = c_sn_impurity[bi_mask]  # Sn trapped in Bi

fig, axs = plt.subplots(1, 4, figsize=(32, 8))

# Plot 1: Raw Structure
axs[0].imshow(phase_matrix, cmap='bone')
axs[0].set_title("1. High-Res Phase Matrix", fontsize=14)
axs[0].axis('off')

# Plot 2: THE COMPOSITIONAL GRADIENT
im1 = axs[1].imshow(composition_map, cmap='viridis', vmax=0.22)
axs[1].set_title("2. Thermodynamic Composition Gradient", fontsize=14)
axs[1].axis('off')
fig.colorbar(im1, ax=axs[1], fraction=0.046, pad=0.04, label="Impurity Atomic Fraction")

# Plot 3: Resistivity Penalty
im2 = axs[2].imshow(rho_map, cmap='inferno')
axs[2].set_title(r"3. Dynamic $\rho(\Delta C)$ Resistivity Map", fontsize=14)
axs[2].axis('off')
fig.colorbar(im2, ax=axs[2], fraction=0.046, pad=0.04, label="Local Resistivity")

# Plot 4: Electrical Flow
im3 = axs[3].imshow(V_matrix, cmap='magma')
axs[3].set_title(f"4. Voltage Field (Global $\\rho_{{eff}}$ = {eff_resistivity:.1f})", fontsize=14)
axs[3].axis('off')
fig.colorbar(im3, ax=axs[3], fraction=0.046, pad=0.04, label="Voltage")

plt.tight_layout()
plt.show()


# Code 11 3D Baker

In [ ]:
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as splinalg
import matplotlib.pyplot as plt
import time
from numba import njit
from scipy.ndimage import convolve

# ==========================================
# 1. PHYSICS & HIGH-RES GRID PARAMETERS
# ==========================================
GRID_SIZE = 1000
VOL_FRAC_SN = 0.49
INTERFACE_PENALTY = 45.0

# EMPIRICAL BASELINE RESISTIVITIES
RHO_SN_BASE = 18.0
RHO_BI_BASE = 129.0

# Baseline Solid Solubilities at Room Temp
BASE_SOL_BI_IN_SN = 0.03
BASE_SOL_SN_IN_BI = 0.001

# Boundary Segregation Limits (High Temp / Trapping)
MAX_SEGREGATION_BI_IN_SN = 0.21
MAX_SEGREGATION_SN_IN_BI = 0.02

# Nordheim Scattering Coefficients (k)
K_BI_IN_SN = 20.0
K_SN_IN_BI = 10000.0

PHASE_BI = 0
PHASE_SN = 1

KMC_STEPS = 50
KT = 0.65
J = 1.0

print(f"Initializing {GRID_SIZE}x{GRID_SIZE} random melt...")
np.random.seed(42)
phase_matrix = np.where(np.random.rand(GRID_SIZE, GRID_SIZE) > (1-VOL_FRAC_SN), PHASE_SN, PHASE_BI).astype(np.int8)

# ==========================================
# 2. NUMBA KMC ENGINE (Quenching the Melt)
# ==========================================
@njit
def run_kmc_growth(matrix, steps, kT, J):
    rows, cols = matrix.shape
    N = rows * cols
    exp_table = np.zeros(9)
    for dE in range(9):
        exp_table[dE] = np.exp(-dE / kT)

    for step in range(steps):
        for _ in range(N):
            x = np.random.randint(0, rows)
            y = np.random.randint(0, cols)

            direction = np.random.randint(0, 4)
            nx, ny = x, y
            if direction == 0: nx = (x + 1) % rows
            elif direction == 1: nx = (x - 1) % rows
            elif direction == 2: ny = (y + 1) % cols
            elif direction == 3: ny = (y - 1) % cols

            current_state = matrix[x, y]
            new_state = matrix[nx, ny]

            if current_state == new_state: continue

            E_initial, E_final = 0, 0
            for dx in (-1, 0, 1):
                for dy in (-1, 0, 1):
                    if dx == 0 and dy == 0: continue
                    nnx = (x + dx) % rows
                    nny = (y + dy) % cols
                    neighbor_state = matrix[nnx, nny]
                    if neighbor_state != current_state: E_initial += J
                    if neighbor_state != new_state: E_final += J

            dE = E_final - E_initial
            if dE <= 0:
                matrix[x, y] = new_state
            else:
                if np.random.rand() < exp_table[int(dE)]:
                    matrix[x, y] = new_state

    return matrix

print(f"Growing Grains via KMC ({KMC_STEPS} steps)...")
start_kmc = time.time()
phase_matrix = run_kmc_growth(phase_matrix, KMC_STEPS, KT, J)
print(f"Grain growth completed in {time.time() - start_kmc:.2f} seconds.")

# ==========================================
# 3. THERMODYNAMIC COMPOSITIONAL GRADIENTS (FICK'S 2ND LAW)
# ==========================================
print("Calculating Thermodynamic Composition Gradients via Fick's 2nd Law...")
start_diff = time.time()

bi_mask = (phase_matrix == PHASE_BI)
sn_mask = (phase_matrix == PHASE_SN)

# Initialize concentration fields at their room-temp baselines
c_sn_impurity = np.full((GRID_SIZE, GRID_SIZE), BASE_SOL_SN_IN_BI)
c_bi_impurity = np.full((GRID_SIZE, GRID_SIZE), BASE_SOL_BI_IN_SN)

# 2D Laplacian Kernel to calculate spatial gradients (∇²C)
laplacian_kernel = np.array([[0,  1,  0],
                             [1, -4,  1],
                             [0,  1,  0]])

# Detect interfacial grain boundaries
sn_edges = np.logical_and(sn_mask, convolve(sn_mask.astype(int), laplacian_kernel, mode='constant') < 0)
bi_edges = np.logical_and(bi_mask, convolve(bi_mask.astype(int), laplacian_kernel, mode='constant') < 0)

# Thermodynamic Diffusion Parameters
D_bi = 0.15
D_sn = 0.10
dt = 0.1
diffusion_steps = 150

# Run Fick's Second Law PDE
for _ in range(diffusion_steps):
    nabla2_c_bi = convolve(c_bi_impurity, laplacian_kernel, mode='nearest')
    nabla2_c_sn = convolve(c_sn_impurity, laplacian_kernel, mode='nearest')

    c_bi_impurity[sn_mask] += (D_bi * nabla2_c_bi * dt)[sn_mask]
    c_sn_impurity[bi_mask] += (D_sn * nabla2_c_sn * dt)[bi_mask]

    c_bi_impurity[sn_edges] = MAX_SEGREGATION_BI_IN_SN
    c_sn_impurity[bi_edges] = MAX_SEGREGATION_SN_IN_BI

c_sn_impurity[bi_mask] += np.random.normal(0, 0.0002, size=np.sum(bi_mask))
c_bi_impurity[sn_mask] += np.random.normal(0, 0.0005, size=np.sum(sn_mask))

print(f"Diffusion gradients resolved in {time.time() - start_diff:.2f} seconds.")

# ==========================================
# 4. VECTORIZED DYNAMIC RESISTIVITY
# ==========================================
print("Applying Nordheim's Rule (Delta-C Offset)...")
rho_map = np.zeros((GRID_SIZE, GRID_SIZE), dtype=float)

delta_c_bi_in_sn = np.maximum(0, c_bi_impurity[sn_mask] - BASE_SOL_BI_IN_SN)
delta_c_sn_in_bi = np.maximum(0, c_sn_impurity[bi_mask] - BASE_SOL_SN_IN_BI)

rho_map[sn_mask] = RHO_SN_BASE + (K_BI_IN_SN * delta_c_bi_in_sn)
rho_map[bi_mask] = RHO_BI_BASE + (K_SN_IN_BI * delta_c_sn_in_bi)

# ==========================================
# 5. FINITE DIFFERENCE SOLVER (Original Safe Method)
# ==========================================
print("Building 1-Million Node Conductance Matrix...")
start_build = time.time()
N = GRID_SIZE * GRID_SIZE

rho1_h, rho2_h = rho_map[:, :-1], rho_map[:, 1:]
p1_h, p2_h = phase_matrix[:, :-1], phase_matrix[:, 1:]
penalty_h = np.where(p1_h != p2_h, INTERFACE_PENALTY, 0.0)
g_h = 1.0 / ((rho1_h + rho2_h) / 2.0 + penalty_h)

rho1_v, rho2_v = rho_map[:-1, :], rho_map[1:, :]
p1_v, p2_v = phase_matrix[:-1, :], phase_matrix[1:, :]
penalty_v = np.where(p1_v != p2_v, INTERFACE_PENALTY, 0.0)
g_v = 1.0 / ((rho1_v + rho2_v) / 2.0 + penalty_v)

main_diag_2D = np.zeros((GRID_SIZE, GRID_SIZE))
main_diag_2D[:, :-1] += g_h
main_diag_2D[:, 1:] += g_h
main_diag_2D[:-1, :] += g_v
main_diag_2D[1:, :] += g_v
main_diag = main_diag_2D.flatten()

g_h_padded = np.zeros((GRID_SIZE, GRID_SIZE))
g_h_padded[:, :-1] = g_h
right_diag = -g_h_padded.flatten()[:-1]
bottom_diag = -g_v.flatten()

offsets = [0, 1, GRID_SIZE]
A = sp.diags([main_diag, right_diag, bottom_diag], offsets, shape=(N, N), format='lil')
A = A + A.T - sp.diags(A.diagonal())
A = A.tolil()

b = np.zeros(N)
for y in range(GRID_SIZE):
    left_idx = y * GRID_SIZE
    right_idx = y * GRID_SIZE + (GRID_SIZE - 1)
    A[left_idx, :] = 0
    A[left_idx, left_idx] = 1.0
    b[left_idx] = 1.0
    A[right_idx, :] = 0
    A[right_idx, right_idx] = 1.0
    b[right_idx] = 0.0

A = A.tocsr()
print(f"Matrix built in {time.time() - start_build:.2f} seconds.")

# ==========================================
# 6. ITERATIVE SOLVER (Original Safe CG)
# ==========================================
print("Solving PDE using Conjugate Gradient...")
start_solve = time.time()
V, info = splinalg.cg(A, b, rtol=1e-8)
print(f"Solver finished in {time.time() - start_solve:.2f} seconds.")

V_matrix = V.reshape((GRID_SIZE, GRID_SIZE))

dV_right = V_matrix[:, -2] - V_matrix[:, -1]
g_right = g_h[:, -1]
total_current = np.sum(dV_right * g_right)
eff_resistivity_2d = 1.0 / total_current

# ==========================================
# 8. BAKKER'S 3D EXTRAPOLATION (1997 EMT)
# ==========================================
# 1. Calculate baseline conductivities (sigma = 1 / rho)
sigma_matrix = 1.0 / np.mean(rho_map[sn_mask])    # High conductivity (Tin)
sigma_dispersed = 1.0 / np.mean(rho_map[bi_mask]) # Low conductivity (Bismuth)

# 2. Calculate the conductivity ratio (lambda_D / lambda_M in Bakker's paper)
cond_ratio = sigma_dispersed / sigma_matrix

# 3. Calculate volume fraction of the dispersed phase (c_D)
c_D = np.mean(bi_mask)

# 4. Calculate the empirical shape factor 'r'
# Using Bakker Eq. 7 (Valid for dispersed phase less conductive than matrix)
r = 0.93 + (1.0 / (cond_ratio + 1.03))

# 5. Convert absolute 2D resistivity to relative 2D conductivity (f_2D)
eff_conductivity_2d = 1.0 / eff_resistivity_2d
f_2D = eff_conductivity_2d / sigma_matrix

# 6. Apply Bakker's 2D -> 3D Transfer Function (Rearranged Eq. 1)
X = 1.0 - ((1.0 - cond_ratio) * c_D)
f_3D = X - ((X - f_2D) / r)

# 7. Convert relative 3D conductivity back to absolute 3D resistivity
eff_conductivity_3d = f_3D * sigma_matrix
eff_resistivity_3d = 1.0 / eff_conductivity_3d

print(f"\n====================================================")
print(f">>> RAW 2D FDM RESISTIVITY: {eff_resistivity_2d:.2f} micro-Ohm-cm")
print(f">>> BAKKER 3D CORRECTED RESISTIVITY: {eff_resistivity_3d:.2f} micro-Ohm-cm <<<")
print(f"====================================================\n")

# ==========================================
# 7. HIGH-RES VISUALIZATION
# ==========================================
composition_map = np.zeros((GRID_SIZE, GRID_SIZE), dtype=float)
composition_map[sn_mask] = c_bi_impurity[sn_mask]
composition_map[bi_mask] = c_sn_impurity[bi_mask]

fig, axs = plt.subplots(1, 4, figsize=(32, 8))

axs[0].imshow(phase_matrix, cmap='bone')
axs[0].set_title("1. High-Res Phase Matrix", fontsize=14)
axs[0].axis('off')

im1 = axs[1].imshow(composition_map, cmap='viridis', vmax=0.22)
axs[1].set_title("2. Thermodynamic Composition Gradient", fontsize=14)
axs[1].axis('off')
fig.colorbar(im1, ax=axs[1], fraction=0.046, pad=0.04, label="Impurity Atomic Fraction")

im2 = axs[2].imshow(rho_map, cmap='inferno')
axs[2].set_title(r"3. Dynamic $\rho(\Delta C)$ Resistivity Map", fontsize=14)
axs[2].axis('off')
fig.colorbar(im2, ax=axs[2], fraction=0.046, pad=0.04, label="Local Resistivity")

im3 = axs[3].imshow(V_matrix, cmap='magma')
axs[3].set_title(f"4. Voltage Field (3D $\\rho_{{eff}}$ = {eff_resistivity_3d:.1f})", fontsize=14)
axs[3].axis('off')
fig.colorbar(im3, ax=axs[3], fraction=0.046, pad=0.04, label="Voltage")

plt.tight_layout()
plt.show()

In [ ]:
# Google Colab-ready figure export block for Sn-Bi multiscale simulation
# Paste this AFTER your solver finishes and the following arrays already exist:
# phase_matrix, composition_map, rho_map, V_matrix, eff_resistivity_3d

import matplotlib.pyplot as plt
from google.colab import files

# -------------------------------
# HIGH-RES EXPORT SETTINGS
# -------------------------------
DPI = 600
FIGSIZE = (8, 8)


def save_single_map(data, cmap, title, filename, colorbar_label=None, vmin=None, vmax=None):
    fig, ax = plt.subplots(figsize=FIGSIZE)
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=16)
    ax.axis('off')

    if colorbar_label is not None:
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label(colorbar_label)

    plt.tight_layout()
    plt.savefig(filename, dpi=DPI, bbox_inches='tight')
    plt.show()
    plt.close()


# 1) Phase matrix
save_single_map(
    phase_matrix,
    cmap='bone',
    title='High-Resolution Phase Matrix',
    filename='phase_matrix_highres.png'
)

# 2) Composition gradient
save_single_map(
    composition_map,
    cmap='viridis',
    title='Thermodynamic Composition Gradient',
    filename='composition_gradient_highres.png',
    colorbar_label='Impurity Atomic Fraction',
    vmax=0.22
)

# 3) Resistivity map
save_single_map(
    rho_map,
    cmap='inferno',
    title='Dynamic Resistivity Map',
    filename='resistivity_map_highres.png',
    colorbar_label='Local Resistivity (µΩ·cm)'
)

# 4) Voltage field
save_single_map(
    V_matrix,
    cmap='magma',
    title=f'Voltage Field (ρ_eff = {eff_resistivity_3d:.1f} µΩ·cm)',
    filename='voltage_field_highres.png',
    colorbar_label='Voltage'
)

print('All 4 figures saved individually at 600 DPI.')
print('Downloading files...')

for f in [
    'phase_matrix_highres.png',
    'composition_gradient_highres.png',
    'resistivity_map_highres.png',
    'voltage_field_highres.png'
]:
    files.download(f)


# FINAL CODE

In [ ]:
"""
Multiscale Simulation of Electrical Resistivity in Sn-Bi Alloys

Pipeline:
1. KMC-based microstructure generation (Ising model)
2. Fickian diffusion for solute redistribution
3. Phenomenological scattering model (linearized Nordheim-type)
4. Finite difference resistor network (Laplace solver)
5. 2D → 3D correction using Bakker EMT

Author: Advait Karmarkar
"""

import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as splinalg
import matplotlib.pyplot as plt
import time
from numba import njit
from scipy.ndimage import convolve

# ==========================================
# 1. PHYSICS & GRID PARAMETERS
# ==========================================
GRID_SIZE = 1000
VOL_FRAC_SN = 0.49

INTERFACE_PENALTY = 45.0  # phenomenological boundary resistance (Sn–Bi interface)

# Baseline resistivities (µΩ·cm)
RHO_SN_BASE = 18.0
RHO_BI_BASE = 129.0

# Equilibrium solubility limits (room temperature)
BASE_SOL_BI_IN_SN = 0.03
BASE_SOL_SN_IN_BI = 0.001

# Solute trapping limits (interface enrichment)
MAX_SEGREGATION_BI_IN_SN = 0.21
MAX_SEGREGATION_SN_IN_BI = 0.02

# Scattering coefficients (linearized Nordheim-type model)
K_BI_IN_SN = 20.0
K_SN_IN_BI = 10000.0  # strong scattering due to low carrier density in Bi

PHASE_BI = 0
PHASE_SN = 1

KMC_STEPS = 50
KT = 0.65
J = 1.0

print(f"Initializing {GRID_SIZE}x{GRID_SIZE} random melt...")
np.random.seed(42)

# Random liquid initialization (volume fraction based)
phase_matrix = np.where(
    np.random.rand(GRID_SIZE, GRID_SIZE) > (1 - VOL_FRAC_SN),
    PHASE_SN,
    PHASE_BI
).astype(np.int8)

# ==========================================
# 2. KMC MICROSTRUCTURE GENERATION
# ==========================================
@njit
def run_kmc_growth(matrix, steps, kT, J):
    rows, cols = matrix.shape
    N = rows * cols

    # Precompute Boltzmann factors
    exp_table = np.zeros(9)
    for dE in range(9):
        exp_table[dE] = np.exp(-dE / kT)

    for _ in range(steps):
        for _ in range(N):
            x = np.random.randint(0, rows)
            y = np.random.randint(0, cols)

            direction = np.random.randint(0, 4)
            nx, ny = x, y
            if direction == 0: nx = (x + 1) % rows
            elif direction == 1: nx = (x - 1) % rows
            elif direction == 2: ny = (y + 1) % cols
            elif direction == 3: ny = (y - 1) % cols

            current_state = matrix[x, y]
            new_state = matrix[nx, ny]

            if current_state == new_state:
                continue

            E_initial, E_final = 0, 0

            # Compute local interaction energies
            for dx in (-1, 0, 1):
                for dy in (-1, 0, 1):
                    if dx == 0 and dy == 0:
                        continue
                    nnx = (x + dx) % rows
                    nny = (y + dy) % cols
                    neighbor_state = matrix[nnx, nny]

                    if neighbor_state != current_state:
                        E_initial += J
                    if neighbor_state != new_state:
                        E_final += J

            dE = E_final - E_initial

            # Metropolis criterion
            if dE <= 0 or np.random.rand() < exp_table[int(dE)]:
                matrix[x, y] = new_state

    return matrix

print(f"Running KMC grain growth ({KMC_STEPS} steps)...")
start_kmc = time.time()
phase_matrix = run_kmc_growth(phase_matrix, KMC_STEPS, KT, J)
print(f"KMC completed in {time.time() - start_kmc:.2f} s")

# ==========================================
# 3. DIFFUSION (FICK'S SECOND LAW)
# ==========================================
print("Solving diffusion (FTCS scheme)...")
start_diff = time.time()

bi_mask = (phase_matrix == PHASE_BI)
sn_mask = (phase_matrix == PHASE_SN)

# Initialize impurity concentration fields
c_sn_impurity = np.full((GRID_SIZE, GRID_SIZE), BASE_SOL_SN_IN_BI)
c_bi_impurity = np.full((GRID_SIZE, GRID_SIZE), BASE_SOL_BI_IN_SN)

# Discrete Laplacian kernel
laplacian_kernel = np.array([[0, 1, 0],
                             [1, -4, 1],
                             [0, 1, 0]])

# Identify phase boundaries using convolution
sn_edges = np.logical_and(sn_mask,
    convolve(sn_mask.astype(int), laplacian_kernel, mode='constant') < 0)

bi_edges = np.logical_and(bi_mask,
    convolve(bi_mask.astype(int), laplacian_kernel, mode='constant') < 0)

# Diffusion parameters (CFL stable)
D_bi, D_sn = 0.15, 0.10
dt = 0.1
diffusion_steps = 150

# Evolve impurity fields
for _ in range(diffusion_steps):
    nabla2_c_bi = convolve(c_bi_impurity, laplacian_kernel, mode='nearest')
    nabla2_c_sn = convolve(c_sn_impurity, laplacian_kernel, mode='nearest')

    c_bi_impurity[sn_mask] += (D_bi * nabla2_c_bi * dt)[sn_mask]
    c_sn_impurity[bi_mask] += (D_sn * nabla2_c_sn * dt)[bi_mask]

    # Dirichlet pinning at interfaces (solute trapping)
    c_bi_impurity[sn_edges] = MAX_SEGREGATION_BI_IN_SN
    c_sn_impurity[bi_edges] = MAX_SEGREGATION_SN_IN_BI

# Small noise to avoid artificial symmetry
c_sn_impurity[bi_mask] += np.random.normal(0, 0.0002, size=np.sum(bi_mask))
c_bi_impurity[sn_mask] += np.random.normal(0, 0.0005, size=np.sum(sn_mask))

print(f"Diffusion completed in {time.time() - start_diff:.2f} s")

# ==========================================
# 4. SCATTERING MODEL (LINEARIZED NORDHEIM-TYPE)
# ==========================================
print("Applying impurity scattering model...")

rho_map = np.zeros((GRID_SIZE, GRID_SIZE), dtype=float)

# Excess impurity above equilibrium solubility
delta_c_bi_in_sn = np.maximum(0, c_bi_impurity[sn_mask] - BASE_SOL_BI_IN_SN)
delta_c_sn_in_bi = np.maximum(0, c_sn_impurity[bi_mask] - BASE_SOL_SN_IN_BI)

# Linearized scattering model
rho_map[sn_mask] = RHO_SN_BASE + (K_BI_IN_SN * delta_c_bi_in_sn)
rho_map[bi_mask] = RHO_BI_BASE + (K_SN_IN_BI * delta_c_sn_in_bi)

# ==========================================
# 5. TRANSPORT SOLVER (FINITE DIFFERENCE)
# ==========================================
print("Building sparse conductance matrix...")
start_build = time.time()

N = GRID_SIZE * GRID_SIZE

# Horizontal conductance
rho1_h, rho2_h = rho_map[:, :-1], rho_map[:, 1:]
p1_h, p2_h = phase_matrix[:, :-1], phase_matrix[:, 1:]
penalty_h = np.where(p1_h != p2_h, INTERFACE_PENALTY, 0.0)
g_h = 1.0 / ((rho1_h + rho2_h) / 2.0 + penalty_h)

# Vertical conductance
rho1_v, rho2_v = rho_map[:-1, :], rho_map[1:, :]
p1_v, p2_v = phase_matrix[:-1, :], phase_matrix[1:, :]
penalty_v = np.where(p1_v != p2_v, INTERFACE_PENALTY, 0.0)
g_v = 1.0 / ((rho1_v + rho2_v) / 2.0 + penalty_v)

# Build matrix diagonals
main_diag_2D = np.zeros((GRID_SIZE, GRID_SIZE))
main_diag_2D[:, :-1] += g_h
main_diag_2D[:, 1:] += g_h
main_diag_2D[:-1, :] += g_v
main_diag_2D[1:, :] += g_v

main_diag = main_diag_2D.flatten()

g_h_padded = np.zeros((GRID_SIZE, GRID_SIZE))
g_h_padded[:, :-1] = g_h

right_diag = -g_h_padded.flatten()[:-1]
bottom_diag = -g_v.flatten()

A = sp.diags([main_diag, right_diag, bottom_diag],
             [0, 1, GRID_SIZE], shape=(N, N), format='lil')

# Enforce symmetry
A = A + A.T - sp.diags(A.diagonal())
A = A.tolil()

# Boundary conditions (1V → 0V)
b = np.zeros(N)

for y in range(GRID_SIZE):
    left = y * GRID_SIZE
    right = y * GRID_SIZE + (GRID_SIZE - 1)

    A[left, :] = 0
    A[left, left] = 1.0
    b[left] = 1.0

    A[right, :] = 0
    A[right, right] = 1.0
    b[right] = 0.0

A = A.tocsr()
print(f"Matrix built in {time.time() - start_build:.2f} s")

# Solve using Conjugate Gradient
print("Solving transport equation...")
start_solve = time.time()

V, info = splinalg.cg(A, b, rtol=1e-8)

print(f"Solver completed in {time.time() - start_solve:.2f} s")

V_matrix = V.reshape((GRID_SIZE, GRID_SIZE))

# Compute effective resistivity (2D)
dV_right = V_matrix[:, -2] - V_matrix[:, -1]
g_right = g_h[:, -1]

total_current = np.sum(dV_right * g_right)
eff_resistivity_2d = 1.0 / total_current

# ==========================================
# 6. 2D → 3D CORRECTION (BAKKER EMT)
# ==========================================
print("Applying 3D correction (Bakker EMT)...")

sigma_matrix = 1.0 / np.mean(rho_map[sn_mask])
sigma_dispersed = 1.0 / np.mean(rho_map[bi_mask])

cond_ratio = sigma_dispersed / sigma_matrix
c_D = np.mean(bi_mask)

# Empirical shape factor
r = 0.93 + (1.0 / (cond_ratio + 1.03))

eff_conductivity_2d = 1.0 / eff_resistivity_2d
f_2D = eff_conductivity_2d / sigma_matrix

X = 1.0 - ((1.0 - cond_ratio) * c_D)
f_3D = X - ((X - f_2D) / r)

eff_conductivity_3d = f_3D * sigma_matrix
eff_resistivity_3d = 1.0 / eff_conductivity_3d

print("\n====================================")
print(f"2D Resistivity: {eff_resistivity_2d:.2f} µΩ·cm")
print(f"3D Corrected Resistivity: {eff_resistivity_3d:.2f} µΩ·cm")
print("====================================\n")

# ==========================================
# 7. VISUALIZATION
# ==========================================
composition_map = np.zeros((GRID_SIZE, GRID_SIZE))
composition_map[sn_mask] = c_bi_impurity[sn_mask]
composition_map[bi_mask] = c_sn_impurity[bi_mask]

fig, axs = plt.subplots(1, 4, figsize=(32, 8))

axs[0].imshow(phase_matrix, cmap='bone')
axs[0].set_title("Phase Map")
axs[0].axis('off')

im1 = axs[1].imshow(composition_map, cmap='viridis', vmax=0.22)
axs[1].set_title("Impurity Distribution")
axs[1].axis('off')
fig.colorbar(im1, ax=axs[1])

im2 = axs[2].imshow(rho_map, cmap='inferno')
axs[2].set_title("Resistivity Map")
axs[2].axis('off')
fig.colorbar(im2, ax=axs[2])

im3 = axs[3].imshow(V_matrix, cmap='magma')
axs[3].set_title(f"Voltage Field (ρ = {eff_resistivity_3d:.1f})")
axs[3].axis('off')
fig.colorbar(im3, ax=axs[3])

plt.tight_layout()
plt.show()